# EW-RIE*: Làm sạch ma trận tương quan có trọng số mũ bằng lý thuyết ma trận ngẫu nhiên
### Ứng dụng: danh mục phương sai nhỏ nhất trên HOSE

(Tóm tắt kết quả: điền sau khi chạy.)

## Phân công nhóm

| Mã | Việc | Đầu ra | Mục trong notebook |
|---|---|---|---|
| **T1** ✔ | Dữ liệu HOSE 2015–11/2026 (vnstock/VCI) + **mã đã hủy niêm yết** để loại survivorship bias | `data_hose/` | §2 |
| **T2** ✔ | Độ vững: chi phí GD thực tế (phí + thuế 0,1%), tần suất tái cân bằng 5/21/63 phiên, xử lý phiên không giao dịch | Bảng độ vững | §6, T2 |
| **T3** ✔ | Đối chứng & kiểm định: 1/N, mô hình nhân tố (PCA), DCC-GARCH; kiểm định phương sai Ledoit–Wolf (2011), Sharpe (Jobson–Korkie/Memmel) | Bảng kiểm định | §6, T3 |
| **T4** | Lý thuyết: chọn nhánh, điều kiện ngoại lai, hiệu chỉnh N hữu hạn; rà soát tính mới (BBP 2016, Phys. Rep. 2017, Oriol 2024) | Methodology (LaTeX) | — |
| **T5** ✔ | Vì sao VS giúp trên HOSE? | Thí nghiệm | §6, T5 |
| **T6** ✔ | Mở rộng: N = 50/100/150, cửa sổ T, half-life | Bảng phụ lục | §6, T6 |
| **T7** | Viết bài, nhắm tạp chí (kiểm tra hạng Q trên Scimago) | Bản thảo | — |

**Khác bản mẫu của thầy:** (1) dữ liệu `data_hose/` thay cho bộ GitHub 2006–3/2020; (2) lọc mã đứng giá `ZERO_MAX` khi chọn universe; (3) sửa EW-NL-LW khi N > T_eff và thêm đối chứng EW-WeSpeR (Oriol 2024); (4) cấu hình chuẩn chọn lại theo lưới T6. Các ô thư viện của thầy giữ nguyên.

## 0. Cài đặt & cấu hình

In [ ]:
import os, sys, subprocess, time, warnings, hashlib, json, platform, importlib.util
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.ticker as mtick
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB, "| numpy", np.__version__, "| pandas", pd.__version__)

# ---- kiểu hình chung: nền trắng, bỏ khung trên/phải, lưới mờ, một bảng màu cố định cho mọi hình ----
PAL = {"EW-RIE*": "#1baf7a", "EW-WeSpeR": "#8e44ad", "EW-NL-LW": "#eb6834", "NL-LW": "#2a78d6",
       "VS-NL-LW": "#0b8a96", "VS-RIE": "#7cc6cc", "RIE": "#9db7e8", "EW-NL-LW (v2)": "#eda100"}
GREY, INK = "#c3c6cc", "#3a3a3a"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
                     "axes.titlelocation": "left", "axes.labelsize": 9, "axes.edgecolor": "#9a9a9a", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": .8,
                     "axes.axisbelow": True, "legend.frameon": False, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
                     "figure.constrained_layout.use": True})
col = lambda k: PAL.get(k, GREY)

In [ ]:
# ======================= CONFIG =======================
CONFIG = dict(
    N=100,            # số mã thanh khoản nhất mỗi kỳ
    WINDOW=500,       # cửa sổ ước lượng (phiên)
    REB=21,           # tái cân bằng mỗi REB phiên
    COST=0.0025,      # chi phí 1 chiều (T2c: phí thực tế mua 0,15%/bán 0,25% ≡ 0,20%)
    HL_EW=60,         # half-life trọng số mũ
    HL_VOL=20,        # half-life chuẩn hóa biến động (VS)
    MAX_ABS_RET=0.15, # |r| > ngưỡng coi là lỗi dữ liệu (biên độ HOSE ±7%)
    ZERO_MAX=0.20,    # mới: loại mã có > 20% phiên r = 0 trong cửa sổ quá khứ (1.0 = không lọc, như bản thầy)
    QUICK=False,      # True: chạy nhanh với ít bộ ước lượng
)
DATA_SOURCE = "csv"   # dữ liệu data_hose/ (HOSE 2015–2026, gồm mã hủy niêm yết)
DATA_DIR = next((d for d in ("../data_hose", "./data_hose") if os.path.exists(f"{d}/prices.csv")),
                "https://raw.githubusercontent.com/chinguyenkhac/data_HOSE/main/data_hose")
WORKDIR = os.environ.get("RMT_WORKDIR", "/content/rmt" if IN_COLAB else "./rmt_work")
os.makedirs(WORKDIR, exist_ok=True)
CONFIG

## 1. Thư viện: các bộ ước lượng
Không cần sửa. Mỗi bộ ước lượng nhận ma trận lợi suất `X` (T×N) và trả về ma trận hiệp phương sai N×N.
Hai thay đổi so với bản thầy: `nl_lw_S` có nhánh N > T_eff (Ledoit–Wolf 2020, p > n) và thêm EW-WeSpeR (ô riêng).

In [ ]:
# ---------- tiện ích ----------
def cov_to_corr(S):
    sd = np.sqrt(np.diag(S)); return S / np.outer(sd, sd), sd

def renorm(C):
    d = np.sqrt(np.diag(C)); return C / np.outer(d, d)

# ---------- đối chứng cổ điển ----------
def est_sample(X): return np.cov(X, rowvar=False)

def est_lw(X): return LedoitWolf().fit(X).covariance_

# ---------- v3: NL-LW cho cả trường hợp p > n (N > T_eff) ----------
def nl_lw_S(S, n):
    # Ledoit & Wolf (2020), co phi tuyến giải tích; S: hiệp phương sai, n: cỡ mẫu (hiệu dụng, có thể không nguyên).
    # p <= n: công thức gốc (giống hệt v2).  p > n: nhánh "p > n" trong mã gốc analytical_shrinkage.m của LW (2020):
    # chỉ ⌊n⌋ trị riêng lớn nhất được xem là "khác 0" (dùng cho mật độ nhân), các trị riêng còn lại nhận chung d̃₀
    # (d̃₀ dựa trên Hilbert transform tại 0). v2 dùng công thức p <= n với c = p/n > 1 ⇒ mẫu số (1 − c − πcλH)² sai, co sai các trị riêng nhỏ.
    p = len(S); lam, U = np.linalg.eigh(S); lam = np.maximum(lam, 1e-12)
    m = p if p <= n else max(int(np.floor(n)), 2)
    lz = lam[p - m:]; h = n ** (-1 / 3)
    L = np.tile(lz, (m, 1)).T; H = h * L.T; x = (L - L.T) / H
    ft = (3 / 4 / np.sqrt(5)) * np.mean(np.maximum(1 - x**2 / 5, 0) / H, axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        Hf = (-3/10/np.pi) * x + (3/4/np.sqrt(5)/np.pi) * (1 - x**2/5) * np.log(np.abs((np.sqrt(5) - x) / (np.sqrt(5) + x)))
    e = np.isclose(np.abs(x), np.sqrt(5)); Hf[e] = (-3/10/np.pi) * x[e]
    Ht = np.mean(Hf / H, axis=1); c = p / n
    if p <= n:
        d = lz / ((np.pi*c*lz*ft)**2 + (1 - c - np.pi*c*lz*Ht)**2)
    else:
        Hf0 = (1/np.pi) * (3/10/h**2 + 3/4/np.sqrt(5)/h * (1 - 1/5/h**2) * np.log((1 + np.sqrt(5)*h) / (1 - np.sqrt(5)*h))) * np.mean(1 / lz)
        d0 = 1 / (np.pi * (p - n) / n * Hf0)
        d1 = lz / (np.pi**2 * lz**2 * (ft**2 + Ht**2))
        d = np.concatenate([np.full(p - m, d0), d1])
    return (U * d) @ U.T

def nl_lw_S_v2(S, n):
    # BẢN v2 (chỉ để đối chiếu ở Test 2/4 và T6b): luôn dùng công thức p <= n, kể cả khi c = p/n > 1
    p = len(S); lam, U = np.linalg.eigh(S); lam = np.maximum(lam, 1e-12)
    L = np.tile(lam, (p, 1)).T; H = n ** (-1 / 3) * L.T; x = (L - L.T) / H
    ft = (3 / 4 / np.sqrt(5)) * np.mean(np.maximum(1 - x**2 / 5, 0) / H, axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        Hf = (-3/10/np.pi) * x + (3/4/np.sqrt(5)/np.pi) * (1 - x**2/5) * np.log(np.abs((np.sqrt(5) - x) / (np.sqrt(5) + x)))
    e = np.isclose(np.abs(x), np.sqrt(5)); Hf[e] = (-3/10/np.pi) * x[e]
    Ht = np.mean(Hf / H, axis=1); c = p / n
    d = lam / ((np.pi*c*lam*ft)**2 + (1 - c - np.pi*c*lam*Ht)**2)
    return (U * d) @ U.T

def est_nllw(X):
    Xc = X - X.mean(0); n = len(X) - 1
    return nl_lw_S(Xc.T @ Xc / n, n)

def mp_corr(C, q):
    lam, V = np.linalg.eigh(C); N = len(C)
    lp = (1 - lam[-1] / N) * (1 + np.sqrt(q))**2
    noise = lam < lp; lam = lam.copy()
    if noise.any(): lam[noise] = lam[noise].mean()
    return renorm((V * lam) @ V.T)

def rie_corr(C, q):
    # RIE Bun–Bouchaud–Potters (2016) + hiệu chỉnh N hữu hạn (2017, §8). Đây là bản GỐC (đối chứng).
    N = len(C); lam, V = np.linalg.eigh(C); lam = np.maximum(lam, 1e-10)
    z = lam - 1j / np.sqrt(N); g = (1 / (z[:, None] - lam[None, :])).mean(1)
    xi = lam / np.abs(1 - q + q*z*g)**2
    s2 = lam[0] / (1 - np.sqrt(q))**2 if q < 1 else lam.mean()
    lp, lm = s2 * (1 + np.sqrt(q))**2, s2 * (1 - np.sqrt(q))**2
    gmp = (z + s2*(q - 1) - np.sqrt(z - lm) * np.sqrt(z - lp)) / (2*q*z*s2)
    xi = xi * np.maximum(1, s2 * np.abs(1 - q + q*z*gmp)**2 / lam)
    xi *= N / xi.sum(); return renorm((V * xi) @ V.T)

def _corr_wrap(cleaner):
    def est(X):
        T, N = X.shape; C, sd = cov_to_corr(np.cov(X, rowvar=False))
        return cleaner(C, N / T) * np.outer(sd, sd)
    return est
est_mp, est_rie = _corr_wrap(mp_corr), _corr_wrap(rie_corr)

In [ ]:
# ---------- EW (trọng số mũ) và VS (chuẩn hóa biến động) ----------
def ew_weights(T, hl):
    w = 0.5 ** ((T - 1 - np.arange(T)) / hl); return w / w.sum()

def ew_corr(X, w):
    Xc = X - w @ X; S = (Xc * w[:, None]).T @ Xc
    sd = np.sqrt(np.diag(S)); return S / np.outer(sd, sd), sd

def make_ew(cleaner, hl):
    # cleaner(C, w) -> tương quan đã làm sạch
    def est(X):
        w = ew_weights(len(X), hl); C, sd = ew_corr(X, w)
        return cleaner(C, w) * np.outer(sd, sd)
    return est

def ewma_standardize(X, hl):
    lam = 0.5 ** (1 / hl); s2 = X[:hl].var(0) + 1e-10; Z = np.empty_like(X)
    for t in range(len(X)):
        Z[t] = X[t] / np.sqrt(s2); s2 = lam * s2 + (1 - lam) * X[t]**2
    return Z, np.sqrt(s2)

def make_vs(corr_cleaner, hl_vol):
    # corr_cleaner(Z) -> ma trận tương quan đã làm sạch của lợi suất chuẩn hóa Z
    def est(X):
        Z, sig = ewma_standardize(X, hl_vol); Z = Z[hl_vol:]
        return corr_cleaner(Z) * np.outer(sig, sig)
    return est

In [ ]:
# ---------- EW-RIE*: RIE chính xác cho trọng số mũ (phương pháp đề xuất) ----------
_STATS = {"fail": 0, "total": 0}; _NULL_CACHE = {}

def _F(g, t, a, w):
    d = 1 - a[None, :] * g[:, None]; R = (w[None, :] / d).sum(1)
    return g * R - t, R, (w[None, :] * a[None, :] / d**2).sum(1)

def _solve_gamma(t, a, w, g0, iters=30):
    # Newton có tìm kiếm lùi cho  γ·R_W(γ) = t  (vector phức), a = N·w
    g = g0.copy(); f, R, dR = _F(g, t, a, w)
    for _ in range(iters):
        step = f / (R + g * dR); lam_ = np.ones(len(g))
        for _ in range(30):
            gn = g - lam_ * step; fn, Rn, dRn = _F(gn, t, a, w)
            bad = ~(np.abs(fn) < np.abs(f)) & (np.abs(f) > 1e-14)
            if not bad.any(): break
            lam_[bad] *= 0.5
        keep = bad | ~np.isfinite(fn)
        gn[keep], fn[keep], Rn[keep], dRn[keep] = g[keep], f[keep], R[keep], dR[keep]
        g, f, R, dR = gn, fn, Rn, dRn
        if np.max(np.abs(f)) < 1e-13: break
    return g, np.abs(f)

def _continuation(t, a, w, steps=60, strict=True):
    # Nhánh chính: γ ≈ t khi t→0, đi theo tia s·t (s: 0→1)
    g = 1e-3 * t
    for s in np.geomspace(1e-3, 1.0, steps):
        g, res = _solve_gamma(s * t, a, w, g)
    ok = np.isfinite(g) & (res < 1e-8)
    if strict and not ok.all(): raise RuntimeError("Newton không hội tụ")
    return g if strict else (g, ok)

def _xi(spec, x, w, N, eta):
    z = x - 1j * eta; g = (1 / (z[:, None] - spec[None, :])).mean(1); t = z * g - 1
    gam, ok = _continuation(t, N * w, w, strict=False)
    gam[~ok] = (t / (1 + N * (w**2).sum() * t))[~ok]       # dự phòng (thực tế: 0 lần trên HOSE)
    _STATS["fail"] += int((~ok).sum()); _STATS["total"] += len(ok)
    return x * gam.imag / t.imag

def _xi_outliers(lam, idx, w, N):
    a = N * w; out = np.empty(len(idx))
    for k, i in enumerate(idx):
        g = np.sum(1 / (lam[i] - np.delete(lam, i))) / N     # Stieltjes bỏ-một trên trục thực
        t = np.array([lam[i] * g - 1 + 0j])
        try:
            gam = _continuation(t, a, w); _, R, dR = _F(gam, t, a, w)
            out[k] = lam[i] * np.real(1 / (R + gam * dR))[0]   # ξ = λ·γ'(t)
        except RuntimeError:
            out[k] = _xi(lam, lam[i:i+1], w, N, 1 / np.sqrt(N))[0]
    return out

def _null_spectrum(N, w, reps=8, seed=0):
    key = (N, len(w), round(float(w[-1]), 12))
    if key not in _NULL_CACHE:
        rng = np.random.default_rng(seed)
        ev = [np.linalg.eigvalsh((Y * w[:, None]).T @ Y) for Y in (rng.standard_normal((len(w), N)) for _ in range(reps))]
        _NULL_CACHE[key] = np.sort(np.concatenate(ev))
    return _NULL_CACHE[key]

def ew_rie_corr(C, w, debias=True):
    N = len(C); lam, V = np.linalg.eigh(C); lam = np.maximum(lam, 1e-10); eta = 1 / np.sqrt(N)
    null = _null_spectrum(N, w); s2 = np.median(lam) / np.median(null)
    D = lam[:, None] - lam[None, :]
    dens = (eta / (D**2 + eta**2)).sum(1) - 1 / eta            # mật độ lân cận (bỏ tự thân)
    out = np.where(dens < 0.1 / eta)[0]; bulk = np.setdiff1d(np.arange(N), out)
    xi = np.empty(N)
    xi[bulk] = _xi(lam, lam[bulk], w, N, eta)
    if len(out): xi[out] = _xi_outliers(lam, out, w, N)
    if debias and len(bulk):
        xi_null = _xi(null, lam[bulk] / s2, w, N, eta)
        xi[bulk] *= np.maximum(1, 1 / np.maximum(xi_null, 1e-6))
    xi = np.maximum(xi, 1e-8); xi *= N / xi.sum()
    return renorm((V * xi) @ V.T)

In [ ]:
# ---------- v3: EW-WeSpeR – đối chứng kiểu QuEST cho hiệp phương sai có trọng số (Oriol 2024, arXiv:2410.14413) ----------
# Mô hình: E = Σ_t w_t x_t x_tᵀ, x_t = C^{1/2} y_t, Σw = 1. Với r(e) = Σ_t w_t / (1 + N w_t e) (= R_W(−e)):
#   e(z) = (1/N) Σ_i τ_i / (τ_i r(e(z)) − z),      m(z) = (1/N) Σ_i 1 / (τ_i r(e(z)) − z)     (Im z > 0)
# m là Stieltjes của phổ mẫu tiệm cận (mật độ ρ = Im m / π); e = (1/N) tr C·G(z). Co phi tuyến "oracle":
#   d(λ) = lim u_λᵀ C u_λ = Im e(λ + i0) / Im m(λ + i0)      (trọng số đều ⇒ λ / |1 − q − qλ m|², Ledoit–Péché).
# WeSpeR = (1) tìm phổ quần thể τ sao cho phân vị của ρ_τ khớp trị riêng mẫu (như QuEST, nhưng với trọng số),
#          (2) áp d(λ_i) với τ đã khớp. Khác EW-RIE*: EW-RIE* dùng trực tiếp Stieltjes thực nghiệm (làm trơn η = N^{-1/2}),
#          WeSpeR dùng Stieltjes của MÔ HÌNH đã khớp ⇒ không cần η, không cần xử lý ngoại lai/hiệu chỉnh N hữu hạn riêng.
# Ghi chú: cài lại bằng numpy từ các phương trình trên (bản gốc của tác giả viết bằng PyTorch + autodiff; phiên làm việc này
# không truy cập được arXiv). τ tham số hóa: log τ tuyến tính từng khúc theo hạng (K nút) + các ngoại lai tự do;
# tối ưu bằng least_squares (Levenberg–Marquardt kiểu trf), sai số tương đối log λ̂_i − log λ_i; khởi tạo ấm từ kỳ trước.
_WSP = {"nfev": [], "cost": [], "warm": {}}

def _wsp_bins(w, nb=40):
    # nén phân phối trọng số thành nb nhóm (r chỉ phụ thuộc phân phối của w): giá trị TB và khối lượng Σw mỗi nhóm
    ws = np.sort(w); g = np.array_split(ws, min(nb, len(ws)))
    return np.array([x.mean() for x in g]), np.array([x.sum() for x in g])

def _wsp_solve(z, tau, wv, wm, N, e0=None, iters=40):
    # Newton (có tìm kiếm lùi, giữ Im e > 0) cho F(e) = e − mean τ/(τ r(e) − z), vector theo z
    def F(e):
        den = 1 + N * wv[None, :] * e[:, None]
        r = (wm[None, :] / den).sum(1); dr = -(N * wm[None, :] * wv[None, :] / den**2).sum(1)
        q = tau[None, :] * r[:, None] - z[:, None]
        f = e - (tau[None, :] / q).mean(1)
        df = 1 + (tau[None, :]**2 * dr[:, None] / q**2).mean(1)
        return f, df, q
    e = (tau[None, :] / (tau[None, :] - z[:, None])).mean(1) if e0 is None else e0.copy()
    f, df, q = F(e)
    for _ in range(iters):
        st = f / df; a = np.ones(len(e))
        for _ in range(25):
            en = e - a * st; fn, dfn, qn = F(en)
            bad = (~np.isfinite(fn)) | (np.abs(fn) > np.abs(f)) | (en.imag <= 0)
            bad &= np.abs(f) > 1e-13
            if not bad.any(): break
            a[bad] *= 0.5
        keep = bad
        en[keep], fn[keep], dfn[keep], qn[keep] = e[keep], f[keep], df[keep], q[keep]
        e, f, df, q = en, fn, dfn, qn
        if np.max(np.abs(f)) < 1e-11: break
    return e, (1 / q).mean(1), np.abs(f)

def _wsp_cold(x, eta, tau, wv, wm, N, steps=14):
    # tiếp tục theo η: từ η lớn (e ≈ mean τ/(τ − z)) xuống η đích ⇒ đúng nhánh Stieltjes (Im e > 0)
    e = None
    for s in np.geomspace(10 * max(x.max(), tau.max()), eta.min(), steps):
        e, m, res = _wsp_solve(x + 1j * np.maximum(eta, s), tau, wv, wm, N, e)
    return _wsp_solve(x + 1j * eta, tau, wv, wm, N, e)

def _wsp_quantiles(x, m, n_nz, N):
    # phân vị của phần khối lượng dương của ρ = Im m/π trên lưới x (hình thang), khớp với n_nz trị riêng khác 0
    rho = np.maximum(m.imag, 0) / np.pi; cdf = np.concatenate([[0], np.cumsum(0.5 * (rho[1:] + rho[:-1]) * np.diff(x))])
    cdf /= cdf[-1]; u = (np.arange(n_nz) + 0.5) / n_nz
    return np.interp(u, cdf, x)

def _wsp_fwd(x, eta, tau, wv, wm, N, e0=None):
    # giải e trên lưới (ấm nếu có e0, nếu không/không hội tụ thì tiếp tục theo η) + các đạo hàm theo τ (đạo hàm ẩn)
    ok = False
    if e0 is not None:
        e, m, res = _wsp_solve(x + 1j * eta, tau, wv, wm, N, e0, iters=15); ok = np.isfinite(e).all() and res.max() < 1e-9
    if not ok: e, m, res = _wsp_cold(x, eta, tau, wv, wm, N)
    return e, m, res

def _wsp_jac_m(x, eta, e, tau, wv, wm, N):
    # dm/dτ_j (G×N):  F(e,τ) = e − mean τ/q, q = τ r(e) − z
    z = x + 1j * eta; den = 1 + N * wv[None, :] * e[:, None]
    r = (wm[None, :] / den).sum(1); dr = -(N * wm[None, :] * wv[None, :] / den**2).sum(1)
    q = tau[None, :] * r[:, None] - z[:, None]; q2 = q**2
    dF = 1 + (tau[None, :]**2 * dr[:, None] / q2).mean(1)
    de = -(z[:, None] / (N * q2)) / dF[:, None]
    A = -(tau[None, :] * dr[:, None] / q2).mean(1)
    return -(r[:, None] / (N * q2)) + A[:, None] * de

def ew_wesper_corr(C, w, K=8, max_nfev=25, G=350, return_info=False):
    from scipy.optimize import least_squares
    N = len(C); lam, V = np.linalg.eigh(C); T = len(w)
    n_nz = min(N, T - 1); lz = np.maximum(lam[N - n_nz:], 1e-10)          # trị riêng khác 0 (tăng dần)
    wv, wm = _wsp_bins(w)
    eta0 = 1 / np.sqrt(N); D = lz[:, None] - lz[None, :]                    # ngoại lai: như ew_rie_corr
    dens = (eta0 / (D**2 + eta0**2)).sum(1) - 1 / eta0
    n_out = int(min(np.sum(dens < 0.1 / eta0), 5)); nb = N - n_out
    knots = np.linspace(0, 1, K); rk = (np.arange(nb) + 0.5) / nb
    def tau_of(th):
        tb = np.exp(np.interp(rk, knots, np.cumsum(np.concatenate([[th[0]], np.exp(th[1:K])]))))
        return np.concatenate([tb, np.exp(th[K:])]) if n_out else tb
    x = np.unique(np.concatenate([np.geomspace(lz[0] * .25, lz[-1] * 1.6, G),
                                  *[lz[-k] * np.linspace(.6, 1.4, 60) for k in range(1, n_out + 1)]]))
    eta = 2e-3 * x; dx = np.diff(x); u = (np.arange(n_nz) + 0.5) / n_nz
    key = (N, T, K, n_out, round(float(w[-1]), 12))
    qb = np.maximum.accumulate(np.log(np.quantile(lz[:n_nz - n_out] if n_nz > n_out else lz, knots)))
    th_cold = np.concatenate([[qb[0]], np.log(np.maximum(np.diff(qb), 1e-3)), np.log(lz[n_nz - n_out:] * .9) if n_out else []])
    st = {}
    def evaluate(th):
        if st.get("th") is not None and np.array_equal(st["th"], th): return
        tau = tau_of(th); e, m, res = _wsp_fwd(x, eta, tau, wv, wm, N, st.get("e"))
        rho = np.maximum(m.imag, 0) / np.pi; raw = np.concatenate([[0], np.cumsum(0.5 * (rho[1:] + rho[:-1]) * dx)])
        tot = raw[-1]; cdf = raw / tot; qh = np.interp(u, cdf, x)
        st.update(th=th.copy(), tau=tau, e=e, m=m, rho=rho, raw=raw, tot=tot, cdf=cdf, qh=qh, ok=np.isfinite(qh).all())
    def fun(th):
        evaluate(th); return np.log(st["qh"]) - np.log(lz) if st["ok"] else np.full(n_nz, 10.0)
    def jac(th):
        evaluate(th)
        dm = _wsp_jac_m(x, eta, st["e"], st["tau"], wv, wm, N); dr = dm.imag / np.pi
        draw = np.vstack([np.zeros(N), np.cumsum(0.5 * (dr[1:] + dr[:-1]) * dx[:, None], axis=0)])
        dcdf = (draw - st["cdf"][:, None] * draw[-1][None, :]) / st["tot"]
        k = np.clip(np.searchsorted(x, st["qh"]) - 1, 0, len(x) - 2); f = (st["qh"] - x[k]) / dx[k]
        dC = dcdf[k] * (1 - f)[:, None] + dcdf[k + 1] * f[:, None]
        pdf = np.maximum((st["rho"][k] * (1 - f) + st["rho"][k + 1] * f) / st["tot"], 1e-3 / (lz[-1]))
        Jt = -dC / pdf[:, None] / st["qh"][:, None]                              # ∂ log λ̂ / ∂τ
        h = 1e-6; t0 = tau_of(th)
        Tt = np.column_stack([(tau_of(th + h * np.eye(len(th))[j]) - t0) / h for j in range(len(th))])
        return Jt @ Tt
    fit = lambda th0: (st.clear(), least_squares(fun, th0, jac=jac, method="trf", max_nfev=max_nfev, xtol=1e-4, ftol=1e-4))[1]
    sol = fit(_WSP["warm"].get(key, th_cold))                 # khởi tạo ấm từ kỳ trước (cùng N, T, h) nếu có
    if key in _WSP["warm"] and np.sqrt(np.mean(sol.fun**2)) > 0.05:  # ấm hỏng (cực tiểu địa phương) ⇒ khớp lại từ đầu, giữ nghiệm tốt hơn
        s2 = fit(th_cold); sol = s2 if np.sum(s2.fun**2) < np.sum(sol.fun**2) else sol
    _WSP["warm"][key] = sol.x; _WSP["nfev"].append(sol.nfev); _WSP["cost"].append(np.sqrt(np.mean(sol.fun**2)))
    tau = tau_of(sol.x)
    # co: d(λ_i) = Im e / Im m tại λ_i + iη (η nhỏ; ở khe/ngoài giá đỡ tỷ số → e'(λ)/m'(λ), vẫn xác định)
    ee, mm, _ = _wsp_cold(lz, 1e-4 * lz, tau, wv, wm, N); d = ee.imag / mm.imag
    xi = np.empty(N); xi[N - n_nz:] = np.maximum(d, 1e-8)
    if n_nz < N:   # không gian null (N ≥ T): giữ vết (tr C = N) cho các hướng null
        xi[:N - n_nz] = max((N - xi[N - n_nz:].sum()) / (N - n_nz), 1e-3 * np.mean(xi[N - n_nz:]))
    xi *= N / xi.sum(); out = renorm((V * xi) @ V.T)
    return (out, dict(tau=tau, xi=xi, lam=lam, rmse=_WSP["cost"][-1], nfev=sol.nfev, n_out=n_out)) if return_info else out


# Bọc an toàn + cache theo (C, w): cùng cửa sổ ở hai chế độ chỉ ước lượng 1 lần; lỗi số ⇒ dự phòng EW-RIE* (được đếm)
_WSP_SAFE = {"fallback": 0, "total": 0}; _WSP_CACHE = {}
def ew_wesper_safe(C, w):
    key = (hashlib.md5(C.tobytes()).hexdigest(), len(w), round(float(w[-1]), 14))
    if key not in _WSP_CACHE:
        _WSP_SAFE["total"] += 1
        try:
            out = ew_wesper_corr(C, w); assert np.isfinite(out).all()
        except Exception:
            _WSP_SAFE["fallback"] += 1; out = ew_rie_corr(C, w)
        _WSP_CACHE[key] = out
    return _WSP_CACHE[key]

In [ ]:
# ---------- Danh sách bộ ước lượng ----------
def build_estimators(cfg):
    hl, hv = cfg["HL_EW"], cfg["HL_VOL"]
    E = {
        "Sample": est_sample,
        "Ledoit-Wolf": est_lw,
        "NL-LW": est_nllw,
        "MP-clip": est_mp,
        "RIE": est_rie,
        "VS-NL-LW": make_vs(lambda Z: renorm(nl_lw_S(np.cov(Z, rowvar=False), len(Z) - 1)), hv),
        "VS-RIE": make_vs(lambda Z: rie_corr(np.corrcoef(Z, rowvar=False), Z.shape[1] / len(Z)), hv),
        "EW-NL-LW": make_ew(lambda C, w: renorm(nl_lw_S(C, 1 / (w**2).sum())), hl),        # v3: đúng cả khi N > T_eff
        "EW-RIE (q_eff)": make_ew(lambda C, w: rie_corr(C, len(C) * (w**2).sum()), hl),
        "EW-WeSpeR": make_ew(ew_wesper_safe, hl),                                          # v3: đối chứng Oriol (2024)
        "EW-RIE*": make_ew(ew_rie_corr, hl),
    }
    if cfg["QUICK"]:
        E = {k: E[k] for k in ["Sample", "NL-LW", "RIE", "EW-NL-LW", "EW-WeSpeR", "EW-RIE*"]}
    return E

## 1b. Kiểm thử (chạy trước mọi thay đổi)
1) Trọng số đều ⇒ EW-RIE\* trùng RIE cổ điển. 2) So với oracle trên dữ liệu tổng hợp đã biết C thật (thêm h = 15: N = 100 > T_eff ≈ 43).

In [ ]:
rng = np.random.default_rng(0); N_, T_ = 100, 250
X = rng.standard_normal((T_, N_)); Cs = np.corrcoef(X, rowvar=False); lam = np.linalg.eigvalsh(Cs)
z = lam - 0.1j; g = (1 / (z[:, None] - lam[None, :])).mean(1); q = N_ / T_
err = np.max(np.abs(_xi(lam, lam, np.full(T_, 1 / T_), N_, 0.1) - lam / np.abs(1 - q + q*z*g)**2) / lam)
print(f"[Test 1] trọng số đều: sai lệch tương đối max = {err:.1e}"); assert err < 1e-10

In [ ]:
def gmv(S):
    w = np.linalg.solve(S, np.ones(len(S))); return w / w.sum()

def factor_corr(N, rng):
    sec = rng.integers(0, 8, N); B = np.zeros((N, 9))
    B[:, 0] = rng.uniform(.4, .8, N); B[np.arange(N), 1 + sec] = rng.uniform(.2, .5, N)
    return renorm(B @ B.T + np.diag(rng.uniform(.3, .6, N)))

rows = []
for hl in (15, 30, 60):
    w = ew_weights(T_, hl); acc = {}
    for rep in range(6):
        Ct = factor_corr(N_, rng); X = rng.standard_normal((T_, N_)) @ np.linalg.cholesky(Ct).T
        C, _ = ew_corr(X, w); lam, V = np.linalg.eigh(C)
        Co = renorm((V * np.einsum("ij,jk,ki->i", V.T, Ct, V)) @ V.T)    # oracle RIE
        wo = gmv(Ct); best = wo @ Ct @ wo
        for k, Ch in {"EW-Sample": C, "EW-RIE (q_eff)": rie_corr(C, N_ * (w**2).sum()),
                      "EW-NL-LW (v2)": renorm(nl_lw_S_v2(C, 1 / (w**2).sum())), "EW-NL-LW": renorm(nl_lw_S(C, 1 / (w**2).sum())),
                      "EW-WeSpeR": ew_wesper_corr(C, w), "EW-RIE*": ew_rie_corr(C, w), "Oracle": Co}.items():
            wh = gmv(Ch); acc.setdefault(k, []).append(wh @ Ct @ wh / best)
    rows += [dict(hl=hl, method=k, risk_ratio=np.mean(v)) for k, v in acc.items()]
t2 = pd.DataFrame(rows).pivot(index="method", columns="hl", values="risk_ratio").round(3)
teff = {hl: 1 / (ew_weights(T_, hl)**2).sum() for hl in (15, 30, 60)}
print("[Test 2] rủi ro GMV / rủi ro tối ưu (1 = hoàn hảo), mô hình nhân tố, N=100, T=250; T_eff =",
      {h: round(v) for h, v in teff.items()}); display(t2)
assert (t2.loc["EW-RIE*", [30, 60]] < t2.loc["EW-RIE (q_eff)", [30, 60]]).all()
assert (t2.loc["EW-NL-LW"] <= t2.loc["EW-NL-LW (v2)"] + 1e-9).all()                 # bản sửa không kém v2
assert np.isclose(t2.loc["EW-NL-LW", 60], t2.loc["EW-NL-LW (v2)", 60])              # N < T_eff: trùng v2
assert (t2.loc["EW-WeSpeR"] / t2.loc["Oracle"] < 1.03).all()                        # WeSpeR trong 3% của oracle

**Đọc bảng Test 2.** Số trong bảng = rủi ro GMV / rủi ro tối ưu (1 là hoàn hảo). Dòng *Oracle* là cận dưới. Ở h = 30 và 60, EW-RIE\* và EW-WeSpeR nằm sát Oracle. Ở h = 15 (N > T_eff), EW-NL-LW bản cũ (v2) vỡ, bản đã sửa và EW-WeSpeR vẫn sát Oracle, còn EW-RIE\* kém đi.

## 2. Dữ liệu (T1)
Chỉ đổi bộ dữ liệu so với bản thầy: `data_hose/` (vnstock 5.2.2, nguồn VCI), giá đóng cửa điều chỉnh + khối lượng, 01/2015 → 11/2026, **gồm cả mã đã hủy niêm yết**.
`P` = giá, `V` = giá × khối lượng (giá trị GD), `MASTER` = lịch sử niêm yết HOSE (mỗi dòng một giai đoạn niêm yết). Chỉ giữ quan sát trong khoảng niêm yết; ngày đình chỉ/không giao dịch để NaN (không nội suy).

In [ ]:
def load_hose(d):
    P = pd.read_csv(f"{d}/prices.csv", index_col=0, parse_dates=True).sort_index()
    Q = pd.read_csv(f"{d}/volumes.csv", index_col=0, parse_dates=True).reindex(index=P.index, columns=P.columns)
    M = pd.read_csv(f"{d}/security_master.csv", parse_dates=["listing_date", "delisting_date"])
    return P, Q, M, pd.read_csv(f"{d}/missing_tickers.csv")

P_RAW, Q, MASTER, MISSING = load_hose(DATA_DIR)
MASTER = MASTER.drop_duplicates(subset=["ticker", "listing_date", "delisting_date"]).reset_index(drop=True)
INWIN = pd.DataFrame(False, index=P_RAW.index, columns=P_RAW.columns)       # mặt nạ khoảng niêm yết HOSE
for r in MASTER.itertuples():
    if r.ticker in INWIN:
        INWIN.loc[(INWIN.index >= r.listing_date) & ((INWIN.index <= r.delisting_date) if pd.notna(r.delisting_date) else True), r.ticker] = True
OUTSIDE = P_RAW.notna() & ~INWIN
P = P_RAW.where(INWIN); Q = Q.where(INWIN); V = P * Q
P = P[P.notna().sum(1) >= 20]; V = V.reindex(P.index); Q = Q.reindex(P.index)
DELISTED = set(MASTER.loc[MASTER.delisting_date.notna(), "ticker"]) - set(MASTER.loc[MASTER.delisting_date.isna(), "ticker"])
LAST_DELIST = MASTER.dropna(subset=["delisting_date"]).groupby("ticker").delisting_date.max()
print(f"{P.shape[1]} mã có giá, {P.shape[0]} phiên: {P.index[0].date()} → {P.index[-1].date()} | {MASTER.ticker.nunique()} mã từng niêm yết, "
      f"{len(DELISTED)} mã đã rời HOSE, {len(MISSING)} mã thiếu giá | quan sát ngoài khoảng niêm yết (đã loại): {int(OUTSIDE.sum().sum())}")

### T1a – Lịch sử niêm yết và mã hủy niêm yết

In [ ]:
days = P.index; listed = pd.Series(0, index=days)
for r in MASTER.itertuples():
    listed[(days >= r.listing_date) & ((days <= r.delisting_date) if pd.notna(r.delisting_date) else True)] += 1
withpx = P.notna().sum(1)
RN = {"bat_buoc": "bắt buộc", "hop_nhat": "hợp nhất/sáp nhập", "tu_nguyen": "tự nguyện", "chuyen_san": "chuyển sàn", "chua_ro": "chưa rõ"}
dl = MASTER.dropna(subset=["delisting_date"]); dl = dl[dl.delisting_date >= P.index[0]]
DY = dl.assign(year=dl.delisting_date.dt.year, reason=dl.reason.map(RN).fillna("chưa rõ")).pivot_table(
    index="year", columns="reason", values="ticker", aggfunc="count", fill_value=0)
DY = DY[DY.sum().sort_values(ascending=False).index]
NL = MASTER[MASTER.listing_date >= P.index[0]].listing_date.dt.year.value_counts().sort_index()
T1A = pd.DataFrame({"Niêm yết mới": NL, "Hủy niêm yết": DY.sum(1),
                    "Hủy – có giá": dl[dl.ticker.isin(P.columns)].delisting_date.dt.year.value_counts(),
                    "Hủy – thiếu giá": dl[~dl.ticker.isin(P.columns)].delisting_date.dt.year.value_counts()}).fillna(0).astype(int)
T1A.loc["Tổng"] = T1A.sum(); display(T1A.T)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw=dict(width_ratios=[1.35, 1]))
ax[0].fill_between(days, withpx, listed, color="#eb6834", alpha=.18, lw=0, label="niêm yết nhưng không có giá")
ax[0].plot(listed, color=INK, lw=1.8, label="đang niêm yết (security_master)")
ax[0].plot(withpx, color="#2a78d6", lw=.9, label="có giá trong ngày")
ax[0].set_title("(a) Số mã trên HOSE theo ngày"); ax[0].set_ylabel("số mã"); ax[0].legend(loc="lower right")
bottom = np.zeros(len(DY)); x = np.arange(len(DY))
for c, cc in zip(DY.columns, ["#eb6834", "#2a78d6", "#8e44ad", "#1baf7a", "#9a9a9a"]):
    ax[1].bar(x, DY[c], .7, bottom=bottom, color=cc, label=c); bottom += DY[c].values
ax[1].plot(x, NL.reindex(DY.index).fillna(0).values, color=INK, marker="o", ms=4, lw=1.4, label="niêm yết mới")
ax[1].set_xticks(x, DY.index, rotation=45); ax[1].set_title("(b) Rời HOSE theo năm và lý do"); ax[1].set_ylabel("số mã"); ax[1].legend(ncol=2)
ax[1].grid(axis="x", visible=False); plt.show()

**Hình T1a.** (a) Đường đen: số mã đang niêm yết theo `security_master`; đường xanh: số mã có giá trong ngày; vùng cam: mã niêm yết nhưng không có giá (đình chỉ, không khớp lệnh, hoặc VCI không còn dữ liệu). (b) Cột chồng: số mã rời HOSE mỗi năm theo lý do; đường đen: số mã niêm yết mới. Năm nào cũng có mã rời sàn, phần lớn là hủy bắt buộc.

### T1b – Kiểm tra chất lượng và độ phủ dữ liệu

In [ ]:
R_all = np.log(P).diff(); r_s = np.expm1(R_all); nobs = int(R_all.notna().sum().sum()); inw = INWIN.reindex(P.index)
QA = pd.Series({
    "Số phiên / số mã có giá / số mã từng niêm yết": f"{len(P)} / {P.shape[1]} / {MASTER.ticker.nunique()}",
    "Mã thiếu giá (đã hủy niêm yết)": f"{len(MISSING)} ({int(MISSING.delisting_date.notna().sum())})",
    "Quan sát ngoài khoảng niêm yết (đã loại)": int(OUTSIDE.sum().sum()),
    "% ô NaN trong khoảng niêm yết": f"{100 * (inw & P.isna()).sum().sum() / inw.sum().sum():.1f}",
    "% phiên r = 0 (đứng giá)": f"{100 * (R_all == 0).sum().sum() / nobs:.1f}",
    "% phiên |r| > 7,5% (ngoài biên độ)": f"{100 * (r_s.abs() > .075).sum().sum() / nobs:.3f}",
    "Phiên |r| > 15% (đặt = 0) / khối lượng = 0 mà có giá / giá ≤ 0":
        f"{int((R_all.abs() > CONFIG['MAX_ABS_RET']).sum().sum())} / {int(((Q == 0) & P.notna()).sum().sum())} / {int((P <= 0).sum().sum())}"})
display(QA.to_frame("giá trị"))

# số mã đủ điều kiện mỗi kỳ tái cân bằng theo quy tắc chọn universe (phủ ≥ 95% + có giá phiên trước) và mức lọc đứng giá
R_q = np.expm1(R_all.mask(R_all.abs() > CONFIG["MAX_ABS_RET"], 0.0)); Wn, el = CONFIG["WINDOW"], []
for s in range(Wn + 1, len(P) - 1, CONFIG["REB"]):
    win = R_q.iloc[s - Wn:s]; ok = (win.notna().mean() >= .95) & P.iloc[s - 1].notna(); zs = (win == 0).mean()
    el.append(dict(date=P.index[s], **{"không lọc": ok.sum(), "r=0 ≤ 20% (chuẩn)": (ok & (zs <= .2)).sum(), "r=0 ≤ 10%": (ok & (zs <= .1)).sum()}))
ELIG = pd.DataFrame(el).set_index("date")

cov_m = P.notna().resample("ME").mean().T; first = P.apply(lambda s: s.first_valid_index()); cov_m = cov_m.loc[first.sort_values().index]
nan_y = (inw & P.isna()).groupby(P.index.year).sum().sum(1) / inw.groupby(P.index.year).sum().sum(1) * 100
z_y = (R_all == 0).groupby(P.index.year).sum().sum(1) / R_all.notna().groupby(P.index.year).sum().sum(1) * 100
zr_t = (R_all == 0).sum() / R_all.notna().sum(); mv = V.median()
liq_q = pd.qcut(mv.rank(method="first"), 5, labels=["Q1\nkém nhất", "Q2", "Q3", "Q4", "Q5\ntốt nhất"])

fig, ax = plt.subplots(2, 2, figsize=(13, 7.2))
a = ax[0, 0]; im = a.imshow(cov_m.values, aspect="auto", cmap="Blues", interpolation="nearest", vmin=0, vmax=1)
yr = cov_m.columns.year; tk = [i for i in range(len(yr)) if i == 0 or yr[i] != yr[i - 1]]
a.set_xticks(tk, yr[tk], rotation=45); a.set_yticks([]); a.grid(False); a.set_ylabel(f"{P.shape[1]} mã (xếp theo ngày có giá đầu tiên)")
dlm = [i for i, t in enumerate(cov_m.index) if t in DELISTED]; a.scatter(np.full(len(dlm), -2), dlm, s=2, color="#eb6834", clip_on=False)
fig.colorbar(im, ax=a, shrink=.85, label="tỷ lệ phiên có giá trong tháng"); a.set_title("(a) Độ phủ theo tháng (chấm cam: mã đã rời HOSE)")
a = ax[0, 1]; yy = nan_y.index
a.bar(yy - .2, nan_y.values, .4, color="#eb6834", label="% ô NaN trong khoảng niêm yết"); a.bar(yy + .2, z_y.values, .4, color="#2a78d6", label="% phiên r = 0")
a.set_xticks(yy, yy, rotation=45); a.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0)); a.legend(); a.grid(axis="x", visible=False)
a.set_title("(b) Phiên thiếu giá và phiên đứng giá theo năm")
a = ax[1, 0]; grp = [zr_t[liq_q == q].values * 100 for q in liq_q.cat.categories]
bp = a.boxplot(grp, tick_labels=list(liq_q.cat.categories), patch_artist=True, widths=.55, showfliers=False, medianprops=dict(color=INK))
for b_, c_ in zip(bp["boxes"], plt.cm.Blues(np.linspace(.35, .85, 5))): b_.set_facecolor(c_); b_.set_edgecolor("#7a7a7a")
a.axhline(100 * CONFIG["ZERO_MAX"], color="#eb6834", ls="--", lw=1.2, label=f"ZERO_MAX = {CONFIG['ZERO_MAX']:.0%}")
a.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0)); a.set_ylabel("% phiên r = 0 của mỗi mã"); a.legend(loc="upper right")
a.set_title("(c) Đứng giá theo nhóm thanh khoản (GTGD trung vị)"); a.grid(axis="x", visible=False)
a = ax[1, 1]
for c_, cc in zip(ELIG.columns, ["#9a9a9a", "#1baf7a", "#eda100"]): a.plot(ELIG[c_], color=cc, lw=1.8, label=c_)
a.axhline(CONFIG["N"], color="#eb6834", ls="--", lw=1.2, label=f"N = {CONFIG['N']}"); a.set_ylabel("số mã"); a.legend(ncol=2)
a.set_title("(d) Số mã đủ điều kiện mỗi kỳ tái cân bằng"); plt.show()
print(f"Tương quan hạng (GTGD, % r=0) = {mv.rank().corr(zr_t.rank()):.2f} | số kỳ có < N mã đủ ĐK:",
      {c_: f"{(ELIG[c_] < CONFIG['N']).sum()}/{len(ELIG)}" for c_ in ELIG})

**Hình T1b.** (a) Mỗi dòng là một mã, màu đậm = có giá đủ phiên trong tháng; mã mới niêm yết xuất hiện dần theo đường chéo, vạch trắng ngang là giai đoạn đình chỉ, chấm cam đánh dấu mã đã rời HOSE. (b) Tỷ lệ ô thiếu giá ổn định 4–9%/năm, tỷ lệ phiên đứng giá cao hơn nhiều (10–24%/năm). (c) Mã càng kém thanh khoản càng hay đứng giá; đường cam là ngưỡng lọc 20%. (d) Số mã đủ điều kiện vào universe theo từng mức lọc; đường cam là N, kỳ nào nằm dưới đường này thì bị bỏ.

## 3. Danh mục & backtest
Vũ trụ động: mỗi kỳ chọn N mã có giá trị GD trung vị cao nhất trong cửa sổ (≥ 95% phiên có dữ liệu, có giá phiên trước, **tỷ lệ phiên r = 0 ≤ `ZERO_MAX`** — dòng duy nhất thêm vào hàm của thầy). Mã ngừng GD/hủy niêm yết ngoài mẫu → lợi suất 0.

In [ ]:
def gmv_unconstrained(S):
    w = np.linalg.solve(S, np.ones(len(S))); return w / w.sum()

def gmv_long_only(S, iters=3000):
    # FISTA + chiếu lên đơn hình (khớp SLSQP, nhanh hơn)
    n = len(S)
    def proj(v):
        u = np.sort(v)[::-1]; css = np.cumsum(u) - 1
        k = np.nonzero(u - css / np.arange(1, n + 1) > 0)[0][-1]; return np.maximum(v - css[k] / (k + 1), 0)
    L = 2 * np.linalg.eigvalsh(S)[-1]; w = y = np.full(n, 1 / n); t = 1.0
    for _ in range(iters):
        wn = proj(y - 2 * S @ y / L); tn = (1 + np.sqrt(1 + 4*t*t)) / 2
        y = wn + (t - 1) / tn * (wn - w)
        if np.abs(wn - w).max() < 1e-9: return wn
        w, t = wn, tn
    return w

PERIODS = [("2016-17 Ổn định", "2016-01-01", "2017-12-31"), ("2018-19 Điều chỉnh", "2018-01-01", "2019-12-31"),
           ("2020Q1 COVID", "2020-01-01", "2020-03-31"), ("2020-21 Bùng nổ", "2020-04-01", "2021-12-31"),
           ("2022 Sụt giảm", "2022-01-01", "2022-12-31"), ("2023-24 Phục hồi", "2023-01-01", "2024-12-31"),
           ("2025-26", "2025-01-01", "2026-12-31")]
ULOG = []   # log universe mỗi kỳ (v3: có sẵn trong run_backtest)

def run_backtest(P, V, estimators, cfg, long_only):
    N, W, REB = cfg["N"], cfg["WINDOW"], cfg["REB"]
    R = np.log(P).diff(); R = np.expm1(R.mask(R.abs() > cfg["MAX_ABS_RET"], 0.0))
    dates = R.index; names = list(estimators)
    rets = {k: [] for k in names}; idx = []; reb_dates = []
    Wp = {k: pd.Series(dtype=float) for k in names}
    st = {k: dict(turn=[], neff=[], wmax=[], short=[]) for k in names}
    for s in range(W + 1, len(dates) - 1, REB):
        win = R.iloc[s - W:s]; ok = (win.notna().mean() >= .95) & P.iloc[s - 1].notna()
        ok &= (win == 0).mean() <= cfg.get("ZERO_MAX", 1.0)          # v3: lọc mã đứng giá (cửa sổ quá khứ)
        if ok.sum() < N: continue
        tick = V.iloc[s - W:s].loc[:, ok].median().sort_values(ascending=False).index[:N]
        ULOG.append(dict(date=dates[s], n_eligible=int(ok.sum()), tickers=list(tick)))
        X = win[tick].fillna(0).values; Y = R.iloc[s:s + REB][tick].fillna(0).values
        idx.extend(dates[s:s + REB]); reb_dates.append(dates[s])
        for k in names:
            S = estimators[k](X)
            w = gmv_long_only(S) if long_only else gmv_unconstrained(S)
            ws = pd.Series(w, index=tick)
            if len(Wp[k]):
                u = ws.index.union(Wp[k].index)
                st[k]["turn"].append((ws.reindex(u).fillna(0) - Wp[k].reindex(u).fillna(0)).abs().sum())
            Wp[k] = ws; rets[k].extend(Y @ w)
            st[k]["neff"].append(1 / (w**2).sum()); st[k]["wmax"].append(w.max()); st[k]["short"].append(np.clip(-w, 0, None).sum())
    out = {}
    for k in names:
        r = pd.Series(rets[k], index=idx); rn = r.copy()
        rn.loc[reb_dates[1:]] -= cfg["COST"] * np.array(st[k]["turn"])
        out[k] = dict(r=r, r_net=rn, m={
            "Vol %": r.std() * np.sqrt(252) * 100, "LN/năm %": r.mean() * 252 * 100,
            "Sharpe": r.mean() / r.std() * np.sqrt(252), "Sharpe sau phí": rn.mean() / rn.std() * np.sqrt(252),
            "Turnover/kỳ": np.mean(st[k]["turn"]), "Số mã hiệu dụng": np.mean(st[k]["neff"]),
            "Tỷ trọng max %": np.mean(st[k]["wmax"]) * 100, "Short %": np.mean(st[k]["short"]) * 100})
    return out

def block_bootstrap_ci(ra, rb, block=21, B=2000, seed=0):
    # CI 95% cho Vol(a) − Vol(b), %/năm. (T3: bổ sung kiểm định Ledoit–Wolf 2011)
    rng = np.random.default_rng(seed); n = len(ra); nb = n // block; d = np.empty(B)
    for i in range(B):
        ix = (rng.integers(0, n - block, nb)[:, None] + np.arange(block)).ravel()
        d[i] = (ra[ix].std() - rb[ix].std()) * np.sqrt(252) * 100
    return np.percentile(d, [2.5, 97.5])

def summarize(out, pairs):
    tab = pd.DataFrame({k: o["m"] for k, o in out.items()}).T
    per = pd.DataFrame({k: {p: o["r"][s:e].std() * np.sqrt(252) * 100 for p, s, e in PERIODS if len(o["r"][s:e]) > 20}
                        for k, o in out.items()}).T
    ci = pd.DataFrame({f"{a} vs {b}": block_bootstrap_ci(out[a]["r"].values, out[b]["r"].values)
                       for a, b in pairs if a in out and b in out}, index=["CI 2.5%", "CI 97.5%"]).T
    ci["Có ý nghĩa"] = (ci["CI 97.5%"] < 0) | (ci["CI 2.5%"] > 0)
    return tab, per, ci

## 4. Chạy chính
Cấu hình chuẩn ở ô CONFIG. 11 bộ ước lượng × 2 chế độ (bán khống / chỉ mua).

In [ ]:
ESTIMATORS = build_estimators(CONFIG)
PAIRS = [("EW-RIE*", "EW-RIE (q_eff)"), ("EW-RIE*", "EW-NL-LW"), ("EW-RIE*", "EW-WeSpeR"), ("EW-WeSpeR", "EW-NL-LW"), ("EW-RIE*", "NL-LW"),
         ("EW-RIE*", "RIE"), ("NL-LW", "Ledoit-Wolf"), ("VS-NL-LW", "NL-LW")]
RESULTS = {}
for long_only in (False, True):
    mode = "long_only" if long_only else "short_ok"; t0 = time.time()
    out = run_backtest(P, V, ESTIMATORS, CONFIG, long_only)
    RESULTS[mode] = dict(out=out, **dict(zip(["table", "periods", "ci"], summarize(out, PAIRS))))
    print(f"\n===== {mode} | {time.time() - t0:.0f}s =====")
    display(RESULTS[mode]["table"].round(2))
    display(RESULTS[mode]["periods"].round(2))
    display(RESULTS[mode]["ci"].round(3))
print("Newton thất bại:", _STATS, "| WeSpeR dự phòng:", _WSP_SAFE, f"| RMSE log-phân vị WeSpeR TB = {np.mean(_WSP['cost']):.3f}")

In [ ]:
KEY5 = ["EW-RIE*", "EW-WeSpeR", "EW-NL-LW", "NL-LW", "VS-NL-LW"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
for a, mode, ttl in [(ax[0], "short_ok", "(a) Cho phép bán khống"), (ax[1], "long_only", "(b) Chỉ mua")]:
    v = RESULTS[mode]["table"]["Vol %"].sort_values(ascending=False)
    a.barh(v.index, v.values, height=.65, color=[col(k) if k in KEY5 else GREY for k in v.index])
    a.set_xlim(v.min() - .8, v.max() + .45); a.grid(axis="y", visible=False)
    for i, (k, x_) in enumerate(v.items()): a.text(x_ + .03, i, f"{x_:.2f}", va="center", fontsize=8, fontweight="bold" if k == "EW-RIE*" else None)
    a.set_title(f"{ttl}: Vol ngoài mẫu (%/năm)"); a.set_xlabel("Vol %/năm (trục không bắt đầu từ 0)")
for t in ax[0].get_yticklabels() + ax[1].get_yticklabels():
    if t.get_text() == "EW-RIE*": t.set_fontweight("bold")
plt.show()

**Hình 4.1.** Vol ngoài mẫu (%/năm) của 11 bộ ước lượng, xếp từ cao xuống thấp; thanh màu là 5 phương pháp chính (xanh lá: EW-RIE\*, tím: EW-WeSpeR, cam: EW-NL-LW, xanh dương: NL-LW, xanh ngọc: VS-NL-LW), thanh xám là các đối chứng còn lại. Trục hoành không bắt đầu từ 0 nên chênh lệch nhỏ được phóng to.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.9))
for k in KEY5[::-1]:
    o = RESULTS["long_only"]["out"][k]; lw = 2.2 if k == "EW-RIE*" else 1.2
    ax[0].plot(o["r"].rolling(252).std() * np.sqrt(252) * 100, color=col(k), lw=lw, label=k)
    ax[1].plot((1 + o["r_net"]).cumprod(), color=col(k), lw=lw, label=k)
ax[0].set_title("(a) Chỉ mua: Vol trượt 252 phiên (%/năm)"); ax[1].set_title("(b) Chỉ mua: giá trị tích lũy sau phí (1 = đầu kỳ)")
h_, l_ = ax[0].get_legend_handles_labels(); ax[0].legend(h_[::-1], l_[::-1], ncol=2); plt.show()

**Hình 4.2.** (a) Vol trượt 1 năm của danh mục chỉ mua; các đường EW gần trùng nhau, khoảng cách với NL-LW (xanh dương) mở rộng sau các cú sốc 2020, 2022, 2025. (b) Giá trị tích lũy sau phí của cùng 5 danh mục.

In [ ]:
# Lưu kết quả (tải về hoặc chép vào Drive)
with pd.ExcelWriter(f"{WORKDIR}/results_N{CONFIG['N']}_T{CONFIG['WINDOW']}_h{CONFIG['HL_EW']}.xlsx") as xw:
    for mode, d in RESULTS.items():
        d["table"].round(4).to_excel(xw, sheet_name=f"{mode}_tong_hop")
        d["periods"].round(3).to_excel(xw, sheet_name=f"{mode}_giai_doan")
        d["ci"].round(4).to_excel(xw, sheet_name=f"{mode}_CI")
    pd.Series(CONFIG).to_frame("giá trị").to_excel(xw, sheet_name="CONFIG")
print("Đã lưu vào", WORKDIR)

## 5. Độ nhạy half-life (T6 mở rộng thêm N, T)
Gộp vào **T6** (§6): lưới N × T × h, cả hai chế độ, cùng ngày ngoài mẫu.

## 6. Việc của nhóm
Cấu hình chuẩn ở ô CONFIG; mọi so sánh trong một bảng dùng **cùng ngày ngoài mẫu, cùng universe, cùng REB/COST/chế độ** (assert trong `run`). *Gộp* = trước phí; *sau phí* = trừ COST × Σ|Δw| tại ngày tái cân bằng.
Bảng lưu ở `./results/*.csv`, hình ở `./figs/*.png`; backtest được cache ở `./results/cache` (`RECOMPUTE = True` để chạy lại).

In [ ]:
# ---- 6.0 Tiện ích chung cho T1c–T6 (tái dùng run_backtest, summarize, block_bootstrap_ci) ----
from scipy.stats import norm
if importlib.util.find_spec("arch") is None:      # Colab chưa có sẵn `arch` (dùng cho DCC-GARCH ở T3)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arch"], check=True)
SEED = 0; np.random.seed(SEED)
RESULTS_DIR, FIGS_DIR, CACHE = "./results", "./figs", "./results/cache"
for d in (RESULTS_DIR, FIGS_DIR, CACHE): os.makedirs(d, exist_ok=True)
RECOMPUTE = False                       # True: chạy lại mọi backtest; False: dùng cache results/cache/*.pkl nếu có
CORE, MODES = ["NL-LW", "EW-NL-LW", "EW-RIE*"], {"long_only": True, "short_ok": False}
VN = {"long_only": "Chỉ mua", "short_ok": "Bán khống"}
COL = PAL

TIMES = {}
def check_cov(S, name=""):                      # đối xứng, PSD, không NaN
    ev = np.linalg.eigvalsh((S + S.T) / 2)
    assert np.isfinite(S).all() and np.allclose(S, S.T, rtol=0, atol=1e-10 * np.abs(S).max()) and ev[0] > -1e-10 * ev[-1], name
    return ev
def timed(name, f):
    def g(X):
        t = time.perf_counter(); S = f(X); TIMES.setdefault(name, []).append(time.perf_counter() - t); check_cov(S, name); return S
    return g

def run(est, cfg=None, long_only=True, data=None, tag=""):
    # backtest có cache: khóa = tên bộ ước lượng + cfg + chế độ (+ tag dữ liệu). Trả về out, log universe, thời gian, số lần Newton thất bại.
    cfg = cfg or CONFIG; Pd, Vd = data if data is not None else (P, V)
    f = f"{CACHE}/{hashlib.md5(repr((sorted(est), sorted(cfg.items()), long_only) + ((tag,) if tag else ())).encode()).hexdigest()[:12]}.pkl"
    if os.path.exists(f) and not RECOMPUTE: d = pd.read_pickle(f)
    else:
        TIMES.clear(); ULOG.clear(); f0, n0 = _STATS["fail"], _STATS["total"]
        out = run_backtest(Pd, Vd, {k: timed(k, g) for k, g in est.items()}, cfg, long_only)
        d = dict(out=out, ulog=list(ULOG), sec={k: np.mean(v) for k, v in TIMES.items()},
                 newton=(_STATS["fail"] - f0, _STATS["total"] - n0)); pd.to_pickle(d, f)
    idx = [o["r"].index for o in d["out"].values()]
    assert all(i.equals(idx[0]) for i in idx), "common-sample vi phạm"      # cùng ngày OOS cho mọi phương pháp
    return d

net = lambda o, c: o["r"] - (c / CONFIG["COST"]) * (o["r"] - o["r_net"])   # r_net ở mức phí c (phí không đổi tỷ trọng)
ann = lambda r: (r.std() * np.sqrt(252) * 100, r.mean() / r.std() * np.sqrt(252))
def metrics(o, cost=None, dates=None):
    r, rn = o["r"], net(o, cost or CONFIG["COST"])
    if dates is not None: r, rn = r.loc[dates], rn.loc[dates]
    return {"Vol %": ann(r)[0], "Sharpe": ann(r)[1], "Sharpe sau phí": ann(rn)[1], "LN/năm %": r.mean() * 252 * 100,
            "Turnover/kỳ": o["m"]["Turnover/kỳ"], "Số mã hiệu dụng": o["m"]["Số mã hiệu dụng"], "Số ngày OOS": len(r)}

f2 = lambda x, d=2: f"{x:.{d}f}".replace(".", ",")
pv = lambda p: "< 0,001" if p < 1e-3 else f2(p, 3)
md = lambda s: display(Markdown(s))
def show(df, d=3): display(df.style.format(precision=d, decimal=",", thousands=" ", na_rep="–"))
def save(df, name):
    assert np.isfinite(df.select_dtypes("number").values).all(), f"{name}: NaN/inf"; df.to_csv(f"{RESULTS_DIR}/{name}.csv")
def savefig(fig, name): fig.savefig(f"{FIGS_DIR}/{name}.png", bbox_inches="tight"); plt.show()

R_BT = np.log(P).diff(); R_BT = np.expm1(R_BT.mask(R_BT.abs() > CONFIG["MAX_ABS_RET"], 0.0))   # đúng như trong run_backtest
POS = {d: i for i, d in enumerate(R_BT.index)}
BASE = run({k: ESTIMATORS[k] for k in CORE}, CONFIG, True)                 # cấu hình chuẩn, chỉ mua
assert all(np.allclose(BASE["out"][k]["r"].values, RESULTS["long_only"]["out"][k]["r"].values) for k in CORE), "run ≠ §4"
U0 = BASE["ulog"]
ENV = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, arch=__import__("arch").__version__, SEED=SEED, **CONFIG)
json.dump(ENV, open(f"{RESULTS_DIR}/ENV_CONFIG.json", "w"), indent=1)
print(f"Dữ liệu: {P.shape[1]} mã, {P.shape[0]} phiên ({P.index[0].date()} → {P.index[-1].date()}) | OOS: {len(U0)} kỳ, "
      f"{len(BASE['out']['EW-RIE*']['r'])} phiên ({U0[0]['date'].date()} → {BASE['out']['EW-RIE*']['r'].index[-1].date()})")
print(f"Mã đủ điều kiện mỗi kỳ: {min(u['n_eligible'] for u in U0)}–{max(u['n_eligible'] for u in U0)} (N={CONFIG['N']}, ZERO_MAX={CONFIG['ZERO_MAX']:.0%}) | "
      f"run() trùng §4: OK | môi trường: {ENV['python']}, numpy {ENV['numpy']}, pandas {ENV['pandas']}, arch {ENV['arch']}")

### T1c – Survivorship bias
So sánh cấu hình chuẩn trên (i) dữ liệu đầy đủ và (ii) chỉ các mã còn niêm yết đến 11/2026 (như khi chỉ lấy danh sách mã hiện tại).

In [ ]:
SURV = [c for c in P.columns if c not in DELISTED]; DATA_SURV = (P[SURV], V[SURV])
T1C_RUN = {(lab, m): run({k: ESTIMATORS[k] for k in CORE}, CONFIG, lo, data=dat, tag=tg)
           for lab, dat, tg in [("(i) đầy đủ", None, ""), ("(ii) chỉ mã còn niêm yết", DATA_SURV, "survivors")] for m, lo in MODES.items()}
COMMON_S = sorted(set.intersection(*[set(d["out"]["NL-LW"]["r"].index) for d in T1C_RUN.values()]))
T1C = pd.DataFrame([dict(data=lab, mode=m, method=k, **metrics(o, dates=COMMON_S)) for (lab, m), d in T1C_RUN.items() for k, o in d["out"].items()])
save(T1C, "T1c_survivorship")
tb = T1C.pivot_table(index=["mode", "method"], columns="data", values=["Vol %", "LN/năm %"])
for c in ("Vol %", "LN/năm %"): tb[(c, "Δ (ii) − (i)")] = tb[(c, "(ii) chỉ mã còn niêm yết")] - tb[(c, "(i) đầy đủ")]
print(f"Bảng T1c – {len(COMMON_S)} ngày OOS chung; {len(DELISTED)} mã đã rời HOSE bị loại ở (ii)"); show(tb.sort_index(axis=1).rename(index=VN, level=0), 2)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for a, m in zip(ax, MODES):
    L_ = T1C[T1C["mode"] == m].pivot(index="method", columns="data", values="Vol %").reindex(CORE); x = np.arange(len(CORE))
    for j, (c, cc) in enumerate(zip(L_.columns, ["#2a78d6", "#eda100"])):
        a.bar(x + (j - .5) * .36, L_[c], .34, color=cc, label=c)
        for i, v in enumerate(L_[c]): a.text(i + (j - .5) * .36, v + .03, f"{v:.2f}", ha="center", va="bottom", fontsize=7.5)
    a.set_xticks(x, CORE); a.set_ylim(T1C["Vol %"].min() - 1, T1C["Vol %"].max() + .5); a.grid(axis="x", visible=False)
    a.set_title(f"{'(a)' if m == 'long_only' else '(b)'} {VN[m]}: Vol %/năm")
ax[0].legend(loc="upper left"); savefig(fig, "T1c_survivorship")

**Hình T1c.** Vol ngoài mẫu của 3 phương pháp chính trên dữ liệu đầy đủ (xanh) và dữ liệu chỉ gồm mã còn niêm yết (vàng), cùng ngày ngoài mẫu. Cột vàng thấp hơn cột xanh ở mọi phương pháp: bỏ mã đã hủy niêm yết làm rủi ro trông thấp hơn thực tế, nhưng thứ tự giữa các phương pháp không đổi.

### T2 – Độ vững
- **T2a** lưới REB ∈ {5, 21, 63} × COST ∈ {0,15%; 0,25%; 0,40%}.
- **T2b** phiên lợi suất 0: không lọc / lọc > 20% (chuẩn) / lọc > 10% phiên r = 0 trong cửa sổ quá khứ.
- **T2c** quy ước phí: COST đối xứng vs phí thực tế mua 0,15% / bán 0,25%.

In [ ]:
# ---- T2: Độ vững ----
# T2a. Lưới REB ∈ {5, 21, 63} × COST ∈ {0,15%; 0,25%; 0,40%} cho NL-LW, EW-NL-LW, EW-RIE*, cả hai chế độ.
# COST không làm đổi tỷ trọng ⇒ mỗi (REB, chế độ) chạy 1 backtest; Sharpe sau phí ở từng COST tính lại từ chuỗi turnover (kiểm ở T2c).
REBS, COSTS = (5, 21, 63), (0.0015, 0.0025, 0.004)
T2 = {(reb, m): run({k: ESTIMATORS[k] for k in CORE}, dict(CONFIG, REB=reb), lo) for reb in REBS for m, lo in MODES.items()}
T2A = pd.DataFrame([dict(mode=m, REB=reb, COST=c, method=k, **metrics(o, c), **{"Turnover/năm": o["m"]["Turnover/kỳ"] * 252 / reb})
                    for (reb, m), d in T2.items() for k, o in d["out"].items() for c in COSTS])
save(T2A, "T2a_reb_cost_grid")
g = T2A.sort_values("COST").groupby(["method", "mode", "REB"])
assert g["Sharpe sau phí"].apply(lambda s: (s.diff().dropna() <= 1e-12).all()).all(), "COST tăng mà Sharpe sau phí tăng?"

tab = T2A[T2A.COST == CONFIG["COST"]].pivot_table(index=["mode", "REB"], columns="method",
        values=["Vol %", "Sharpe sau phí", "Turnover/năm"])[["Vol %", "Sharpe sau phí", "Turnover/năm"]].reindex(columns=CORE, level=1)
print(f"Bảng T2a – COST = {CONFIG['COST']:.2%} (Vol %/năm gộp; Sharpe sau phí; turnover năm hóa = Σ|Δw|/kỳ × 252/REB)")
show(tab.rename(index=VN, level=0))

fig, ax = plt.subplots(2, 3, figsize=(12.5, 6)); LOD = T2A[T2A["mode"] == "long_only"]
for i, (c, cm) in enumerate([("Vol %", "Blues"), ("Sharpe sau phí", "Greens")]):
    lo_, hi_ = LOD[c].min(), LOD[c].max()
    for j, k in enumerate(CORE):
        Mx = LOD[LOD.method == k].pivot(index="REB", columns="COST", values=c); a = ax[i, j]
        a.imshow(Mx.values, cmap=cm, aspect="auto", vmin=lo_ - .15 * (hi_ - lo_), vmax=hi_); a.grid(False)
        a.set_xticks(range(3), [f"{x_:.2%}".replace(".", ",") for x_ in Mx.columns]); a.set_yticks(range(3), Mx.index)
        for (y, x_), v in np.ndenumerate(Mx.values):
            a.text(x_, y, f2(v), ha="center", va="center", fontsize=9, color="white" if v > lo_ + .6 * (hi_ - lo_) else INK)
        a.set_title(f"{k} — {c}", color=col(k)); a.set_xlabel("COST (1 chiều)" if i else ""); a.set_ylabel("REB (phiên)" if j == 0 else "")
fig.suptitle("T2a – chỉ mua: Vol gộp (trên) và Sharpe sau phí (dưới) theo REB × COST, cùng thang màu giữa 3 phương pháp", fontsize=10)
savefig(fig, "T2a_heatmap")

**Hình T2a.** Mỗi ô là một kịch bản (hàng: tần suất tái cân bằng, cột: chi phí một chiều), chế độ chỉ mua. Hàng trên: Vol gộp — không phụ thuộc COST nên mỗi hàng cùng màu, Vol tăng khi tái cân bằng thưa hơn. Hàng dưới: Sharpe sau phí — giảm khi COST tăng (sang phải). Ba ô cùng hàng dùng chung thang màu nên có thể so trực tiếp giữa các phương pháp.

In [ ]:
# T2b. Phiên lợi suất 0. Code gốc: phiên không khớp lệnh vẫn có giá ⇒ r = 0; NaN trong cửa sổ → fillna(0).
# So sánh (i) gốc với (ii) loại khỏi universe các mã có tỷ lệ r=0 trong CỬA SỔ QUÁ KHỨ > 20% hoặc > 10% (không dùng tương lai).
def zero_share(ulog):    # tỷ lệ r=0 trung bình của universe đã chọn (đúng dữ liệu đưa vào estimator)
    return np.mean([(R_BT.iloc[POS[u["date"]] - CONFIG["WINDOW"]:POS[u["date"]]][u["tickers"]].fillna(0) == 0).values.mean() for u in ulog])
VARS = {"(i) không lọc": 1.0, "(ii) lọc r=0 > 20% (chuẩn v3)": 0.20, "(iii) lọc r=0 > 10%": 0.10}
T2B_RUN = {(v, m): run({k: ESTIMATORS[k] for k in CORE}, dict(CONFIG, ZERO_MAX=z), lo)
           for v, z in VARS.items() for m, lo in MODES.items()}
COMMON = sorted(set.intersection(*[set(d["out"]["NL-LW"]["r"].index) for d in T2B_RUN.values()]))   # kỳ thiếu mã bị bỏ ⇒ so trên ngày chung
T2B = pd.DataFrame([dict(variant=v, mode=m, method=k, **metrics(o, dates=COMMON), **{"Số kỳ hợp lệ": len(d["ulog"]),
                    "Mã đủ ĐK (TB)": np.mean([u["n_eligible"] for u in d["ulog"]]), "Mã đủ ĐK (min)": min(u["n_eligible"] for u in d["ulog"]),
                    "% r=0 trong universe": 100 * zero_share(d["ulog"])}) for (v, m), d in T2B_RUN.items() for k, o in d["out"].items()])
save(T2B, "T2b_zero_return")
print(f"Bảng T2b – Vol %/năm trên {len(COMMON)} ngày OOS chung ({COMMON[0].date()} → {COMMON[-1].date()}); tổng số kỳ chuẩn: {len(U0)}")
tb = T2B.pivot_table(index="variant", columns=["mode", "method"], values="Vol %").reindex(columns=CORE, level=1)
info = T2B[T2B["mode"] == "long_only"].groupby("variant")[["Số kỳ hợp lệ", "Mã đủ ĐK (TB)", "Mã đủ ĐK (min)", "% r=0 trong universe"]].first()
show(pd.concat([tb.rename(columns=VN, level=0), pd.concat({"Universe": info}, axis=1)], axis=1).loc[list(VARS)], 2)
z = tb["long_only"]; base = z.loc[list(VARS)[0]]

fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.6), sharey=True)
for a, m in zip(ax, MODES):
    d = tb[m].loc[list(VARS)]; x = np.arange(len(CORE))
    for j, (vn, cc) in enumerate(zip(VARS, ["#9a9a9a", "#1baf7a", "#eda100"])):
        a.bar(x + (j - 1) * .27, d.loc[vn, CORE], .25, color=cc, label=vn)
        for i, v in enumerate(d.loc[vn, CORE]): a.text(i + (j - 1) * .27, v + .03, f"{v:.1f}", ha="center", va="bottom", fontsize=7)
    a.set_xticks(x, CORE); a.set_ylim(tb.values.min() - 1.5, tb.values.max() + .7); a.grid(axis="x", visible=False)
    a.set_title(f"{'(a)' if m == 'long_only' else '(b)'} {VN[m]}: Vol %/năm theo mức lọc đứng giá")
ax[0].legend(loc="upper left"); savefig(fig, "T2b_zero_filter")

**Hình T2b.** Vol ngoài mẫu (cùng ngày chung) của 3 phương pháp ở ba mức lọc mã đứng giá: không lọc (xám), lọc > 20% (xanh, chuẩn), lọc > 10% (vàng). Lọc càng chặt thì Vol đo được càng cao — mã đứng giá làm rủi ro trông thấp giả tạo — còn thứ tự giữa các phương pháp giữ nguyên.

In [ ]:
# T2c. Quy ước chi phí. run_backtest: phí kỳ = COST × Σ|w_mới − w_cũ| ⇒ COST là phí MỘT CHIỀU áp cho cả mua lẫn bán.
# HOSE: phí môi giới ~0,15% mỗi chiều + thuế bán 0,1% ⇒ bất đối xứng (mua 0,15%, bán 0,25%).
def trade_cost(w_new, w_old, c_buy, c_sell):
    u = w_new.index.union(w_old.index); d = w_new.reindex(u, fill_value=0) - w_old.reindex(u, fill_value=0)
    return c_buy * d.clip(lower=0).sum() + c_sell * (-d).clip(lower=0).sum()
rng = np.random.default_rng(SEED)
for _ in range(200):     # unit test: có mã ra/vào universe và tỷ trọng âm (bán khống), Σw = 1 ở cả hai kỳ
    a_ = pd.Series(rng.normal(size=6), index=list("ABCDEF")); a_ /= a_.sum(); b_ = pd.Series(rng.dirichlet(np.ones(4)), index=list("DEFG"))
    u = a_.index.union(b_.index); turn = (a_.reindex(u, fill_value=0) - b_.reindex(u, fill_value=0)).abs().sum()
    assert np.isclose(trade_cost(a_, b_, .0025, .0025), .0025 * turn) and np.isclose(trade_cost(a_, b_, .0015, .0025), .002 * turn)
o_new = run({"NL-LW": est_nllw}, dict(CONFIG, REB=63, COST=0.004))["out"]["NL-LW"]          # chạy lại thật với COST khác
assert np.allclose(net(T2[63, "long_only"]["out"]["NL-LW"], 0.004).values, o_new["r_net"].values, atol=1e-14)
print("Unit test 200/200 pass: Σw=1 ⇒ giá trị mua = giá trị bán = turnover/2 ⇒ phí (mua 0,15%, bán 0,25%) ≡ COST đối xứng 0,20%.")
print("Kiểm chứng: r_net tính lại từ turnover trùng với chạy lại backtest thật (REB=63, COST=0,40%).")
SC = {"Gộp": None, "COST 0,25% (gốc)": .0025, "Mua 0,15% / bán 0,25%": .002}
T2C = pd.DataFrame([dict(mode=VN[m], method=k, scen=sc, Sharpe=ann(o["r"] if c is None else net(o, c))[1])
                    for m in MODES for k, o in T2[21, m]["out"].items() for sc, c in SC.items()]).pivot(index="method", columns=["mode", "scen"], values="Sharpe")
T2C = T2C.reindex(index=CORE, columns=pd.MultiIndex.from_product([VN.values(), SC]))
save(T2C, "T2c_cost_convention"); print("Sharpe (REB = 21):"); show(T2C)

**Bảng T2c.** Với Σw = 1, giá trị mua bằng giá trị bán nên phí mua 0,15% / bán 0,25% tương đương COST đối xứng 0,20%; giả định gốc 0,25% hơi bảo thủ.

### T3 – Đối chứng & kiểm định
- **T3a** đối chứng mới: 1/N, PCA k nhân tố (k ∈ {1, 3, 5}, PCA-auto chọn k trong cửa sổ huấn luyện), DCC-GARCH(1,1), EW-WeSpeR.
- **T3b** bảng benchmark; **T3c** kiểm định phương sai Ledoit–Wolf (2011) (HAC + studentized block bootstrap, Holm), Sharpe Jobson–Korkie–Memmel; **T3d** hình.

In [ ]:
# ---- T3: Đối chứng & kiểm định ----
# T3a. Đối chứng mới: 1/N, mô hình nhân tố PCA (k ∈ {1,3,5} và tự chọn k trong cửa sổ huấn luyện), DCC-GARCH(1,1).
from arch import arch_model
from scipy.optimize import minimize

def est_equal_weight_proxy(X):
    # 1/N: dùng ma trận đơn vị -> GMV = 1/N (chiến lược phân bổ, KHÔNG phải bộ ước lượng hiệp phương sai)
    return np.eye(X.shape[1])

def est_pca(k):
    # Σ = B Bᵀ + diag(ψ): k thành phần chính của hiệp phương sai mẫu, phần dư chéo (đối xứng, PSD theo cấu trúc)
    def est(X):
        S = np.cov(X, rowvar=False); lam, Q = np.linalg.eigh(S); B = Q[:, -k:] * np.sqrt(lam[-k:])
        return B @ B.T + np.diag(np.maximum(np.diag(S) - (B**2).sum(1), 1e-10))
    return est

PCA_K = []
def est_pca_auto(X, ks=(1, 3, 5), split=0.7):
    # chọn k CHỈ trong cửa sổ huấn luyện: ước lượng trên 70% đầu, đo phương sai GMV trên 30% cuối, rồi ước lượng lại trên cả cửa sổ
    n = int(len(X) * split)
    k = min(ks, key=lambda k: np.var(X[n:] @ gmv_unconstrained(est_pca(k)(X[:n]))))
    PCA_K.append(k); return est_pca(k)(X)

DCC_LOG, _DCC_CACHE = [], {}
def _dcc_ncl(p, Z, Qbar):
    # −composite log-lik (Engle, Shephard & Sheppard 2008): tổng log-lik 2×2 của các cặp kề nhau (i, i+1).
    # Full QML với N=100 vừa chậm vừa kéo a → 0 (thiên lệch đã biết khi N lớn) nên dùng CL.
    a, b = p
    if a < 0 or b < 0 or a + b >= 0.999: return 1e10
    z1, z2 = Z[:, :-1], Z[:, 1:]; q11, q22, q12 = np.diag(Qbar)[:-1], np.diag(Qbar)[1:], np.diag(Qbar, 1)
    c11, c22, c12 = q11.copy(), q22.copy(), q12.copy(); nll = 0.0
    for t in range(len(Z)):
        if t:
            c11 = (1-a-b)*q11 + a*z1[t-1]**2 + b*c11; c22 = (1-a-b)*q22 + a*z2[t-1]**2 + b*c22
            c12 = (1-a-b)*q12 + a*z1[t-1]*z2[t-1] + b*c12
        rho = c12 / np.sqrt(c11 * c22); d = 1 - rho**2
        nll += 0.5 * np.sum(np.log(d) + (z1[t]**2 + z2[t]**2 - 2*rho*z1[t]*z2[t]) / d)
    return nll

def est_dcc(X, h=21):
    # DCC-GARCH(1,1) (Engle 2002): bước 1 GARCH(1,1) đơn biến (`arch`) cho từng mã; bước 2 (a, b) bằng composite
    # likelihood + correlation targeting; ma trận Q_t đầy đủ N×N. Dự báo: D = căn phương sai GARCH trung bình h phiên tới, R = R_{T+1|T}; Σ = D R D.
    key = hash(X.tobytes())
    if key in _DCC_CACHE: return _DCC_CACHE[key]               # cùng cửa sổ ở long_only/short_ok → không ước lượng lại
    T, N = X.shape; Z = np.empty_like(X); hv = np.empty(N); bad = 0
    for i in range(N):
        try:
            r = arch_model(100 * X[:, i], mean="Constant", vol="GARCH", p=1, q=1, rescale=False).fit(disp="off", show_warning=False)
            ok = r.convergence_flag == 0 and r.params["alpha[1]"] + r.params["beta[1]"] < 1
        except Exception:
            ok = False
        if ok:
            Z[:, i] = r.std_resid; hv[i] = r.forecast(horizon=h, reindex=False).variance.values[-1].mean() / 1e4
        else:                                                   # dự phòng: phương sai mẫu (đếm vào DCC_LOG)
            bad += 1; Z[:, i] = (X[:, i] - X[:, i].mean()) / X[:, i].std(); hv[i] = X[:, i].var()
    Qbar = Z.T @ Z / T
    res = minimize(_dcc_ncl, x0=[0.02, 0.95], args=(Z, Qbar), method="Nelder-Mead", options=dict(xatol=1e-4, fatol=1e-3, maxiter=200))
    a, b = res.x if res.fun < 1e9 else (0.0, 0.0)
    Qt = Qbar.copy()
    for t in range(1, T + 1): Qt = (1 - a - b) * Qbar + a * np.outer(Z[t-1], Z[t-1]) + b * Qt    # Q_{T+1}
    d = 1 / np.sqrt(np.diag(Qt)); sd = np.sqrt(hv)
    S = (Qt * np.outer(d, d)) * np.outer(sd, sd)
    DCC_LOG.append(dict(a=a, b=b, garch_fail=bad, dcc_ok=bool(res.success))); _DCC_CACHE[key] = S
    return S

In [ ]:
# T3b. Bảng benchmark: cùng ngày OOS, cùng universe, REB=21, COST=0,25%, hai chế độ.
T3_EST = {"1/N": est_equal_weight_proxy, **{f"PCA-k{k}": est_pca(k) for k in (1, 3, 5)}, "PCA-auto": est_pca_auto,
          "DCC-GARCH(1,1)": est_dcc, "EW-WeSpeR": ESTIMATORS["EW-WeSpeR"], **{k: ESTIMATORS[k] for k in CORE}}
T3R = {m: run(T3_EST, CONFIG, lo) for m, lo in MODES.items()}
T3B = pd.DataFrame([dict(mode=m, method=k, **metrics(o), **{"Tỷ trọng max %": o["m"]["Tỷ trọng max %"], "Giây/lần ước lượng": T3R["long_only"]["sec"][k]})
                    for m, d in T3R.items() for k, o in d["out"].items()])
save(T3B, "T3b_benchmark")
for m in MODES: assert T3B[T3B["mode"] == m].set_index("method")["Turnover/kỳ"].idxmin() == "1/N"   # sanity: 1/N ít giao dịch nhất
tb = T3B.pivot(index="method", columns="mode", values=["Vol %", "Sharpe", "Sharpe sau phí", "Turnover/kỳ", "Số mã hiệu dụng"])
tb.columns = tb.columns.swaplevel(); tb = tb[[c for m in MODES for c in tb.columns if c[0] == m]].sort_values(("long_only", "Vol %"))
tb["Giây/lần ước lượng"] = T3B.drop_duplicates("method").set_index("method")["Giây/lần ước lượng"]
print("Bảng T3b – benchmark (xếp theo Vol chỉ mua; Vol %/năm gộp; Sharpe sau phí trừ COST × turnover)"); show(tb.rename(columns=VN, level=0))
P3 = pd.concat({VN[m]: summarize(d["out"], [])[1] for m, d in T3R.items()}); save(P3, "T3b_periods")
print("Vol %/năm theo giai đoạn thị trường:"); show(P3, 2)
if len(DCC_LOG) >= len(U0):   # log chỉ có khi vừa tính mới (không dùng cache) → lưu CSV; kỳ đầu tiên của chế độ chỉ mua
    pd.DataFrame(DCC_LOG[:len(U0)]).assign(pca_auto_k=PCA_K[:len(U0)]).to_csv(f"{RESULTS_DIR}/T3_dcc_pca_log.csv", index=False)
dl = pd.read_csv(f"{RESULTS_DIR}/T3_dcc_pca_log.csv"); pk = dl.pca_auto_k

In [ ]:
# T3c. Kiểm định: phương sai Ledoit–Wolf (2011) – HAC và studentized circular block bootstrap; Sharpe Jobson–Korkie–Memmel;
# so với block bootstrap CI có sẵn (block_bootstrap_ci), block ∈ {10, 21, 42}; Holm cho 4 so sánh mỗi chế độ.
def hac(u, L=None):
    # HAC Newey–West (Bartlett), L = ⌊4(T/100)^(2/9)⌋. (LW 2008/2011 dùng QS/Parzen + prewhitening – chưa làm.)
    T = len(u); L = L or int(4 * (T / 100) ** (2 / 9)); S = u.T @ u / T
    for l in range(1, L + 1): G = u[l:].T @ u[:-l] / T; S += (1 - l / (L + 1)) * (G + G.T)
    return S

def _lw_stat(y, Psi=None, b=None):
    # Δ = log σ²_a − log σ²_b (LW 2011, delta method); y_t = (r_a, r_b, r_a², r_b²)
    T = len(y); m = y.mean(0); va, vb = m[2] - m[0]**2, m[3] - m[1]**2
    g = np.array([-2 * m[0] / va, 2 * m[1] / vb, 1 / va, -1 / vb])
    if Psi is None:   # Ψ của bootstrap theo khối (LW 2008, công thức (9))
        z = (y - m)[:T // b * b].reshape(-1, b, 4).sum(1) / np.sqrt(b); Psi = z.T @ z / len(z)
    return np.log(va) - np.log(vb), np.sqrt(g @ Psi @ g / T)

def lw_var_test(ra, rb, block=21, M=2000, seed=SEED):
    # p-value HAC (chuẩn) và p-value studentized circular block bootstrap (LW 2008 Remark 3.2, áp cho phương sai như LW 2011)
    y = np.column_stack([ra, rb, ra**2, rb**2]); T = len(y)
    d, se = _lw_stat(y, hac(y - y.mean(0))); p_hac = 2 * norm.sf(abs(d / se))
    rng = np.random.default_rng(seed); l = T // block; stats = np.empty(M)
    for i in range(M):
        ix = ((rng.integers(0, T, l)[:, None] + np.arange(block)) % T).ravel()      # khối vòng
        ds, ses = _lw_stat(y[ix], b=block); stats[i] = abs(ds - d) / ses
    return d, se, p_hac, (np.sum(stats >= abs(d) / se) + 1) / (M + 1)

def jkm_test(ra, rb):
    # Jobson–Korkie (1981) với hiệu chỉnh Memmel (2003); giả định iid chuẩn
    T = len(ra); m1, m2 = ra.mean(), rb.mean(); s1, s2 = ra.std(ddof=1), rb.std(ddof=1); s12 = np.cov(ra, rb)[0, 1]
    th = (2 * s1**2 * s2**2 - 2 * s1 * s2 * s12 + .5 * m1**2 * s2**2 + .5 * m2**2 * s1**2 - m1 * m2 / (s1 * s2) * s12**2) / T
    z = (s2 * m1 - s1 * m2) / np.sqrt(th); return z, 2 * norm.sf(abs(z))

# unit test nhanh: gần trùng ⇒ p lớn; gấp đôi biến động ⇒ p≈0; JKM phản đối xứng, Sharpe chênh lớn ⇒ p≈0
# (JKM suy biến θ=0 khi 2 chuỗi tương quan hoàn hảo – không dùng trường hợp đó)
_r = np.random.default_rng(SEED).standard_normal(1500) * .01
assert lw_var_test(_r, _r + 1e-5 * np.random.default_rng(1).standard_normal(1500), M=200)[3] > .1
assert lw_var_test(_r, 2 * _r[::-1], M=200)[2] < 1e-6                                   # Δ = −log 4
_q = np.random.default_rng(2).standard_normal(1500) * .01
assert np.isclose(jkm_test(_r, _q)[0], -jkm_test(_q, _r)[0]) and jkm_test(_r + .003, _q - .003)[1] < 1e-6

BLOCKS, rows = (10, 21, 42), []
for mode in MODES:
    out = T3R[mode]["out"]
    for b_ in ("NL-LW", "EW-NL-LW", "EW-WeSpeR", "1/N", "PCA-auto"):   # đối chứng định trước, không chọn hậu kiểm
        ra, rb = out["EW-RIE*"]["r"].values, out[b_]["r"].values
        na, nb = (net(out[k], CONFIG["COST"]).values for k in ("EW-RIE*", b_))
        row = dict(mode=mode, pair=f"EW-RIE* vs {b_}", **{"ΔVol (pp)": (ra.std() - rb.std()) * np.sqrt(252) * 100})
        for bl in BLOCKS:
            d, se, p_hac, p_bt = lw_var_test(ra, rb, block=bl)
            lo_, hi_ = block_bootstrap_ci(ra, rb, block=bl)
            row.update({"Δlog σ²": d, "p LW-HAC": p_hac, f"p LW-boot b={bl}": p_bt, f"CI lo b={bl}": lo_, f"CI hi b={bl}": hi_})
        row.update({"p JKM gross": jkm_test(ra, rb)[1], "p JKM sau phí": jkm_test(na, nb)[1],
                    "ΔSharpe gross": ann(pd.Series(ra))[1] - ann(pd.Series(rb))[1]})
        rows.append(row)
T3C = pd.DataFrame(rows)
# multiple comparisons: Holm trên 5 so sánh trong mỗi chế độ (p LW-boot b=21)
T3C["p Holm (LW-boot b=21)"] = T3C.groupby("mode")["p LW-boot b=21"].transform(
    lambda p: np.minimum(1, np.maximum.accumulate(np.sort(p.values) * (len(p) - np.arange(len(p)))))[np.argsort(np.argsort(p.values))])
save(T3C, "T3c_tests")
cols = ["ΔVol (pp)", "CI lo b=21", "CI hi b=21", "p LW-HAC", "p LW-boot b=10", "p LW-boot b=21", "p LW-boot b=42", "p Holm (LW-boot b=21)", "p JKM gross", "p JKM sau phí"]
print("Bảng T3c – EW-RIE* so với đối chứng (ΔVol < 0: EW-RIE* rủi ro thấp hơn; CI = block bootstrap percentile b=21)")
show(T3C.assign(mode=T3C["mode"].map(VN)).set_index(["mode", "pair"])[cols], 4)

In [ ]:
# T3d. Hình: Vol + CI 95% (block bootstrap b=21) dạng điểm; heatmap p-value (các cột chính)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3), sharey=True)
order = pd.Series({k: o["m"]["Vol %"] for k, o in T3R["long_only"]["out"].items()}).sort_values(ascending=False).index
for a, mode in zip(ax, MODES):
    out = T3R[mode]["out"]; v = pd.Series({k: out[k]["m"]["Vol %"] for k in order})
    ci = np.array([block_bootstrap_ci(out[k]["r"].values, np.zeros(len(out[k]["r"]))) for k in v.index]); y = np.arange(len(v))
    a.hlines(y, ci[:, 0], ci[:, 1], color=[col(k) if k in PAL else "#9a9a9a" for k in v.index], lw=2.2)
    a.scatter(v.values, y, s=42, zorder=3, color=[col(k) if k in PAL else "#9a9a9a" for k in v.index], edgecolor="white")
    for i, x_ in enumerate(v.values): a.text(ci[i, 1] + .25, i, f"{x_:.2f}", va="center", fontsize=8)
    a.set_yticks(y, v.index); a.grid(axis="y", visible=False); a.set_xlabel("Vol %/năm (gộp)")
    a.set_title(f"{'(a)' if mode == 'long_only' else '(b)'} {VN[mode]}: Vol và CI 95%")
savefig(fig, "T3_benchmark")

pc = ["p LW-HAC", "p LW-boot b=10", "p LW-boot b=21", "p LW-boot b=42", "p Holm (LW-boot b=21)", "p JKM gross", "p JKM sau phí"]
Mx = T3C.set_index(["mode", "pair"])[pc]
fig, a = plt.subplots(figsize=(10.5, 4.2))
im = a.imshow(-np.log10(Mx.values.clip(1e-4)), cmap="Purples", vmin=0, vmax=4, aspect="auto"); a.grid(False)
a.set_xticks(range(len(pc)), [c[2:] for c in pc], rotation=25, ha="right")
a.set_yticks(range(len(Mx)), [f"{VN[m]} | {p.replace('EW-RIE* vs ', 'vs ')}" for m, p in Mx.index])
for (y, x_), p in np.ndenumerate(Mx.values): a.text(x_, y, f"{p:.3f}", ha="center", va="center", fontsize=8, color="white" if p < .01 else INK)
a.axhline(len(Mx) / 2 - .5, color="white", lw=3)
fig.colorbar(im, ax=a, shrink=.9, label="−log10 p"); a.set_title("T3 – p-value hai phía: EW-RIE* so với từng đối chứng")
savefig(fig, "T3_pvalue_heatmap")

**Hình T3d.** Trên: Vol ngoài mẫu của mọi đối chứng (điểm) với CI 95% block bootstrap của từng chuỗi (đoạn ngang), xếp theo Vol chỉ mua; các khoảng tin cậy riêng lẻ chồng lên nhau, chỉ 1/N tách hẳn. Dưới: p-value của kiểm định *cặp* EW-RIE\* với từng đối chứng — ô tím đậm là p < 0,01; cột LW (phương sai) đo chênh lệch rủi ro, cột JKM đo chênh lệch Sharpe; nửa trên là chỉ mua, nửa dưới là bán khống.

### T5 – Vì sao VS giúp trên HOSE?
- **T5a** ablation chỉ đổi VS: RIE ↔ VS-RIE, NL-LW ↔ VS-NL-LW, theo giai đoạn thị trường.
- **T5b** dự báo biến động 21 phiên: EWMA vs cửa sổ phẳng.
- **T5c** phổ trị riêng trước/sau VS và condition number.

In [ ]:
# ---- T5: Vì sao VS giúp trên HOSE? ----
# T5a. Ablation: mỗi cặp chỉ khác bước chuẩn hóa biến động (VS): RIE ↔ VS-RIE, NL-LW ↔ VS-NL-LW. Dùng lại lần chạy chính §4 (QUICK=False).
ABL = [("RIE", "VS-RIE"), ("NL-LW", "VS-NL-LW")]
rows, per = [], []
for m in MODES:
    out = RESULTS[m]["out"]
    for b, v in ABL:
        lo_, hi_ = block_bootstrap_ci(out[v]["r"].values, out[b]["r"].values)
        mb, mv = metrics(out[b]), metrics(out[v])
        rows.append({"Chế độ": VN[m], "Cặp": f"{v} vs {b}", "Vol gốc": mb["Vol %"], "Vol VS": mv["Vol %"], "ΔVol (pp)": mv["Vol %"] - mb["Vol %"],
                     "CI lo": lo_, "CI hi": hi_, "Sharpe sau phí gốc": mb["Sharpe sau phí"], "Sharpe sau phí VS": mv["Sharpe sau phí"],
                     "Turnover gốc": mb["Turnover/kỳ"], "Turnover VS": mv["Turnover/kỳ"]})
        per.append((RESULTS[m]["periods"].loc[v] - RESULTS[m]["periods"].loc[b]).rename((VN[m], f"{v} − {b}")))
T5A = pd.DataFrame(rows).set_index(["Chế độ", "Cặp"]); T5P = pd.DataFrame(per).T
save(T5A, "T5a_vs_ablation"); save(T5P, "T5a_vs_ablation_periods")
print("Bảng T5a – tác động của VS (Vol %/năm gộp; CI 95% block bootstrap b=21 của ΔVol)"); show(T5A)
print("ΔVol (pp) do VS theo giai đoạn thị trường (< 0: VS giảm rủi ro):"); show(T5P, 2)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
for a, m in zip(ax, VN.values()):
    d = T5P[m]; x = np.arange(len(d))
    for j, (c, cc) in enumerate(zip(d.columns, [PAL["VS-RIE"], PAL["VS-NL-LW"]])): a.bar(x + (j - .5) * .38, d[c], .36, color=cc, label=c)
    a.axhline(0, color=INK, lw=.8); a.set_xticks(x, [s.replace(" ", "\n", 1) for s in d.index]); a.grid(axis="x", visible=False)
    a.set_title(f"{'(a)' if m == 'Chỉ mua' else '(b)'} {m}: ΔVol do VS (pp/năm)")
ax[0].set_ylabel("ΔVol (pp); < 0 ⇒ VS giảm rủi ro"); ax[0].legend(loc="lower left"); savefig(fig, "T5a_vs_periods")

**Hình T5a.** Mỗi cột là chênh lệch Vol (pp/năm) giữa bản có VS và bản gốc trong một giai đoạn thị trường; cột âm = VS giảm rủi ro. Phần lớn lợi ích của VS nằm ở COVID 2020Q1 và 2020–21, các giai đoạn khác chênh lệch nhỏ hoặc dương.

In [ ]:
# T5b. Dự báo phương sai 21 phiên tới: EWMA (hl ∈ {10, 20, 40}, đúng hàm ewma_standardize mà VS dùng) vs cửa sổ phẳng 63/250 phiên.
# Mẫu: các mã trong universe chuẩn ở mỗi kỳ; dự báo chỉ dùng cửa sổ quá khứ. Mục tiêu RV = trung bình r² của 21 phiên sau.
# Sai số: RMSE của vol năm hóa (%) và QLIKE = log h + RV/h (dạng Patton 2011, xác định cả khi RV = 0).
W, H = CONFIG["WINDOW"], 21; rows = []
for u in U0:
    s = POS[u["date"]]
    if s + H > len(R_BT): continue
    X = R_BT.iloc[s - W:s][u["tickers"]].fillna(0).values; rv = (R_BT.iloc[s:s + H][u["tickers"]].fillna(0).values ** 2).mean(0)
    fc = {**{f"EWMA hl={h}": ewma_standardize(X, h)[1] ** 2 for h in (10, 20, 40)}, **{f"Phẳng {w} phiên": (X[-w:] ** 2).mean(0) for w in (63, W)}}
    per_ = next((p for p, a_, b_ in PERIODS if pd.Timestamp(a_) <= u["date"] <= pd.Timestamp(b_)), None)
    for k, h in fc.items():
        h = np.maximum(h, 1e-12); rows.append(dict(period=per_, model=k, se=np.mean((np.sqrt(h) - np.sqrt(rv)) ** 2), ql=np.mean(np.log(h) + rv / h)))
F = pd.DataFrame(rows); F = pd.concat([F, F.assign(period="Toàn mẫu")])
T5B = F.groupby(["period", "model"], sort=False).agg(RMSE=("se", lambda x: np.sqrt(x.mean() * 252) * 100), QLIKE=("ql", "mean"), n=("se", "size"))
save(T5B, "T5b_vol_forecast")
tbl = T5B["RMSE"].unstack()[["EWMA hl=10", "EWMA hl=20", "EWMA hl=40", "Phẳng 63 phiên", f"Phẳng {W} phiên"]]
tbl.loc["QLIKE (toàn mẫu)"] = T5B.xs("Toàn mẫu")["QLIKE"]
print("Bảng T5b – RMSE dự báo vol 21 phiên (%/năm) theo giai đoạn; dòng cuối: QLIKE toàn mẫu (thấp hơn = tốt hơn)"); show(tbl)
Mx = tbl.drop(index="QLIKE (toàn mẫu)"); fig, a = plt.subplots(figsize=(12, 3.6)); x = np.arange(len(Mx))
for j, (c, cc) in enumerate(zip(Mx.columns, ["#9fd8c0", "#1baf7a", "#0d6b4a", "#f0b27a", "#c0392b"])):
    a.bar(x + (j - 2) * .16, Mx[c], .15, color=cc, label=c)
a.set_xticks(x, [s.replace(" ", "\n", 1) for s in Mx.index]); a.set_ylabel("RMSE vol (%/năm)"); a.legend(ncol=5, loc="upper left"); a.grid(axis="x", visible=False)
a.set_title("T5b – Sai số dự báo vol 21 phiên: EWMA (xanh) vs cửa sổ phẳng (cam/đỏ)"); savefig(fig, "T5b_vol_forecast")

**Hình T5b.** RMSE dự báo biến động 21 phiên tới (%/năm) theo giai đoạn; ba sắc xanh là EWMA với half-life 10/20/40, cam và đỏ là cửa sổ phẳng 63/250 phiên. Cột xanh thấp hơn cột cam/đỏ ở hầu hết giai đoạn, khoảng cách lớn nhất ở COVID 2020Q1.

In [ ]:
# T5c. Phổ trị riêng trước/sau VS (1 kỳ ví dụ) + condition number theo kỳ. "Trước": tương quan mẫu của X; "sau": của Z = X/σ_EWMA
# (bỏ HL_VOL phiên đầu, đúng make_vs). Thêm cond của hiệp phương sai THỰC SỰ đưa vào GMV.
hv = CONFIG["HL_VOL"]; rows = []
winX = lambda u: R_BT.iloc[POS[u["date"]] - W:POS[u["date"]]][u["tickers"]].fillna(0).values
for u in U0:
    X = winX(u); Z = ewma_standardize(X, hv)[0][hv:]
    ex, ez = np.linalg.eigvalsh(np.corrcoef(X, rowvar=False)), np.linalg.eigvalsh(np.corrcoef(Z, rowvar=False))
    r = {"date": u["date"], "cond corr(X)": ex[-1] / ex[0], "cond corr(Z), VS": ez[-1] / ez[0], "λ1/N trước VS": ex[-1] / len(ex), "λ1/N sau VS": ez[-1] / len(ez)}
    for k in ("NL-LW", "VS-NL-LW", "RIE", "VS-RIE"): e = check_cov(ESTIMATORS[k](X), k); r[f"cond Σ {k}"] = e[-1] / e[0]
    rows.append(r)
T5C = pd.DataFrame(rows).set_index("date"); save(T5C, "T5c_condition_numbers")
print("Bảng T5c – condition number và trị riêng lớn nhất (trung bình / trung vị trên các kỳ)"); show(T5C.agg(["mean", "median"]).T, 2)
ex_u = next(u for u in U0 if u["date"] >= pd.Timestamp("2020-03-01")); X = winX(ex_u); Z = ewma_standardize(X, hv)[0][hv:]
fig, ax = plt.subplots(1, 2, figsize=(13, 3.7))
for lab, M_, cc in [("trước VS: corr(X)", X, "#9a9a9a"), ("sau VS: corr(Z)", Z, PAL["EW-RIE*"])]:
    ev = np.sort(np.linalg.eigvalsh(np.corrcoef(M_, rowvar=False)))[::-1]; ax[0].plot(np.arange(1, len(ev) + 1), ev, color=cc, lw=1.8, marker="o", ms=2.5, label=lab)
ax[0].axhline((1 + np.sqrt(CONFIG["N"] / W)) ** 2, ls=":", color=INK, label="biên MP λ+ (nhiễu thuần)")
ax[0].set_yscale("log"); ax[0].set_xlabel("thứ hạng trị riêng"); ax[0].legend(); ax[0].set_title(f"(a) Phổ tương quan, kỳ {ex_u['date'].date()}")
for k, cc in [("cond corr(X)", "#9a9a9a"), ("cond corr(Z), VS", PAL["EW-RIE*"]), ("cond Σ NL-LW", PAL["NL-LW"]), ("cond Σ VS-NL-LW", PAL["VS-NL-LW"])]:
    ax[1].plot(T5C.index, T5C[k], color=cc, lw=1.6, label=k)
ax[1].set_yscale("log"); ax[1].legend(ncol=2); ax[1].set_title("(b) Condition number theo kỳ tái cân bằng"); savefig(fig, "T5c_spectrum_condition")

**Hình T5c.** (a) Trị riêng của ma trận tương quan (xếp giảm dần, trục log) trước và sau chuẩn hóa EWMA trong một kỳ đầu 2020; hai đường gần trùng nhau, chỉ vài trị riêng đầu vượt biên nhiễu Marchenko–Pastur. (b) Condition number theo thời gian: tương quan trước/sau VS gần như nhau, còn Σ của VS-NL-LW có condition number cao hơn NL-LW.

### T6 – Khái quát: số tài sản N, cửa sổ T, half-life
Đổi **từng yếu tố**, giữ nguyên phần còn lại của cấu hình chuẩn. So trên ngày ngoài mẫu chung trong mỗi yếu tố. T_eff = 1/Σw², q_eff = N·Σw² (q_eff > 1 ⇔ N > T_eff).

In [ ]:
# ---- T6: đổi TỪNG yếu tố quanh cấu hình chuẩn; NL-LW, EW-NL-LW, EW-RIE*, EW-WeSpeR; cả hai chế độ, ngày OOS chung trong mỗi yếu tố ----
T6K = CORE + ["EW-WeSpeR"]
LEVELS = {"N": sorted({50, 100, 150, CONFIG["N"]}), "WINDOW": sorted({125, 250, 500, CONFIG["WINDOW"]}), "HL_EW": sorted({30, 60, 120, CONFIG["HL_EW"]})}
GRID = [(f, v) for f, vs in LEVELS.items() for v in vs]
T6R = {(f, v, m): run({k: build_estimators(dict(CONFIG, **{f: v}))[k] for k in T6K}, dict(CONFIG, **{f: v}), lo) for f, v in GRID for m, lo in MODES.items()}
rows = []
for f in LEVELS:
    for m in MODES:
        common = sorted(set.intersection(*[set(T6R[f, v, m]["out"]["NL-LW"]["r"].index) for v in LEVELS[f]]))
        for v in LEVELS[f]:
            d = T6R[f, v, m]; o = d["out"]; cfg = dict(CONFIG, **{f: v}); w = ew_weights(cfg["WINDOW"], cfg["HL_EW"])
            ra = o["EW-RIE*"]["r"].loc[common].values
            ci = {b: (lambda rb: ((ra.std() - rb.std()) * np.sqrt(252) * 100, *block_bootstrap_ci(ra, rb)))(o[b]["r"].loc[common].values) for b in ("EW-NL-LW", "EW-WeSpeR", "NL-LW")}
            rows.append(dict(factor=f, value=v, mode=m, q_eff=cfg["N"] * (w ** 2).sum(), **{f"Vol {k}": ann(o[k]["r"].loc[common])[0] for k in T6K},
                             **{f"{s} vs {b}": ci[b][i] for b in ci for i, s in enumerate(("Δ", "lo", "hi"))},
                             **{"Số ngày chung": len(common), "giây/lần EW-RIE*": d["sec"].get("EW-RIE*", np.nan), "giây/lần EW-WeSpeR": d["sec"].get("EW-WeSpeR", np.nan),
                                "Newton fail": d["newton"][0]}))
T6 = pd.DataFrame(rows); save(T6, "T6_generalization")
T6["cấu hình"] = T6.factor.map({"N": "N", "WINDOW": "T", "HL_EW": "h"}) + "=" + T6.value.astype(str)
star = lambda b: T6.apply(lambda r: f2(r[f"Δ vs {b}"]) + ("*" if r[f"hi vs {b}"] < 0 or r[f"lo vs {b}"] > 0 else ""), axis=1)
tb = T6.assign(**{f"Δ vs {b}": star(b) for b in ("NL-LW", "EW-NL-LW", "EW-WeSpeR")}).pivot(
    index="cấu hình", columns="mode", values=[f"Vol {k}" for k in T6K] + ["Δ vs NL-LW", "Δ vs EW-NL-LW", "Δ vs EW-WeSpeR"])
tb.columns = tb.columns.swaplevel(); tb = tb[[c for m in MODES for c in tb.columns if c[0] == m]].reindex(T6.drop_duplicates("cấu hình")["cấu hình"])
print("Bảng T6 – Vol %/năm (ngày OOS chung trong mỗi yếu tố); Δ = Vol(EW-RIE*) − Vol(đối chứng), pp; * = CI 95% không chứa 0")
show(tb.rename(columns=VN, level=0), 2)

LOD = T6[T6["mode"] == "long_only"]; base = {"N": CONFIG["N"], "WINDOW": CONFIG["WINDOW"], "HL_EW": CONFIG["HL_EW"]}
fig, ax = plt.subplots(2, 3, figsize=(13, 6.2), sharex="col", gridspec_kw=dict(height_ratios=[1.25, 1]))
for j, (f, ttl) in enumerate([("N", "số mã N"), ("WINDOW", "cửa sổ T (phiên)"), ("HL_EW", "half-life h (phiên)")]):
    d = LOD[LOD.factor == f].sort_values("value"); x = d.value.values
    for k in T6K: ax[0, j].plot(x, d[f"Vol {k}"], marker="o", ms=4, lw=2.4 if k == "EW-RIE*" else 1.4, color=col(k), label=k, zorder=3 if k == "EW-RIE*" else 2)
    b0 = d[d.value == base[f]]; ax[0, j].scatter(b0.value, b0["Vol EW-RIE*"], s=160, facecolor="none", edgecolor=INK, lw=1.2, zorder=4)
    for b, mk, dx in [("EW-NL-LW", "o", .985), ("EW-WeSpeR", "s", 1.015)]:
        ax[1, j].errorbar(x * dx, d[f"Δ vs {b}"], yerr=[d[f"Δ vs {b}"] - d[f"lo vs {b}"], d[f"hi vs {b}"] - d[f"Δ vs {b}"]],
                          fmt=mk, ms=5, color=col(b), capsize=3, lw=1.6, label=f"vs {b}")
    ax[1, j].axhline(0, color=INK, lw=.8); ax[1, j].set_xscale("log"); ax[1, j].set_xticks(x, x); ax[1, j].minorticks_off(); ax[1, j].set_xlabel(ttl)
    ax[0, j].set_title(f"({'abc'[j]}) Chỉ mua: Vol theo {ttl.split(' (')[0]}")
ax[0, 0].set_ylabel("Vol %/năm"); ax[1, 0].set_ylabel("ΔVol của EW-RIE* (pp), CI 95%"); ax[0, 0].legend(); ax[1, 0].legend()
savefig(fig, "T6_generalization")
print("Thời gian (giây/lần, chỉ mua) EW-RIE*:", f2(LOD["giây/lần EW-RIE*"].median()), "| EW-WeSpeR:", f2(LOD["giây/lần EW-WeSpeR"].median()),
      "| Newton thất bại:", int(T6["Newton fail"].sum()))

**Hình T6.** Hàng trên: Vol chỉ mua của 4 phương pháp khi đổi N, cửa sổ T, half-life h (mỗi cột một yếu tố, các yếu tố khác giữ ở cấu hình chuẩn; điểm khoanh là cấu hình chuẩn). Hàng dưới: chênh lệch Vol của EW-RIE\* so với EW-NL-LW (cam) và EW-WeSpeR (tím), kèm CI 95%; điểm dưới đường 0 nghĩa là EW-RIE\* rủi ro thấp hơn.

### Tự kiểm tra

In [ ]:
# ---- Tự kiểm tra ----
rng_ = np.random.default_rng(0); X_ = rng_.standard_normal((250, 100)); l_ = np.linalg.eigvalsh(np.corrcoef(X_, rowvar=False)); z_ = l_ - .1j
g_ = (1 / (z_[:, None] - l_[None, :])).mean(1); e_ = np.max(np.abs(_xi(l_, l_, np.full(250, 1 / 250), 100, .1) - l_ / np.abs(1 - .4 + .4 * z_ * g_) ** 2) / l_)
checks = {
    "Test 1 chạy lại (thư viện không đổi trạng thái)": e_ < 1e-10,
    "Không NaN/inf trong mọi CSV của results/": all(np.isfinite(pd.read_csv(f"{RESULTS_DIR}/{f}").select_dtypes("number").values).all()
                                                   for f in os.listdir(RESULTS_DIR) if f.endswith(".csv")),
    "run() trùng §4; common-sample; Σ đối xứng/PSD/không NaN (assert trong run/timed)": True,
    "1/N có turnover thấp nhất": all(T3B[T3B["mode"] == m].set_index("method")["Turnover/kỳ"].idxmin() == "1/N" for m in MODES),
    "REB tăng ⇒ turnover/kỳ tăng, turnover/năm giảm": all(
        (T2A[(T2A.COST == CONFIG["COST"]) & (T2A["mode"] == m) & (T2A.method == k)].sort_values("REB")[c].diff().dropna() * s > 0).all()
        for m in MODES for k in CORE for c, s in (("Turnover/kỳ", 1), ("Turnover/năm", -1))),
    f"Newton của EW-RIE* không hội tụ: {_STATS['fail']} lần": _STATS["fail"] == 0,
    f"EW-WeSpeR dự phòng sang EW-RIE*: {_WSP_SAFE['fallback']}/{_WSP_SAFE['total']} lần": _WSP_SAFE["fallback"] == 0,
    "Lọc ZERO_MAX chỉ dùng cửa sổ quá khứ": all(
        ((R_BT.iloc[POS[u["date"]] - CONFIG["WINDOW"]:POS[u["date"]]][u["tickers"]] == 0).mean() <= CONFIG["ZERO_MAX"]).all() for u in U0)}
for k, v in checks.items(): print(("✔ " if v else "✘ ") + k)
assert all(checks.values())

## Ghi chú
- **Khai báo dùng AI:** mã nguồn được xây dựng với hỗ trợ của trợ lý AI; nhóm cần rà soát, hiểu và khai báo theo quy định của tạp chí.
- **Tính mới:** công thức tổng quát cho nhiễu nhân có trong Bun–Bouchaud–Potters (2016, 2017). Oriol (2024, WeSpeR) giải cùng bài toán theo hướng khôi phục phổ quần thể; T4/T7 cần định vị EW-RIE\* so với WeSpeR (công thức đóng, nhanh hơn ≈ 10–25 lần, ngang về rủi ro).
- **EW-WeSpeR** ở đây là bản cài lại bằng numpy theo các phương trình của bài báo, chưa đối chiếu mã gốc. **EW-NL-LW** dùng nhánh p > n của Ledoit–Wolf (2020) khi N > T_eff.
- **Chọn cấu hình chuẩn:** N, T, h trong ô CONFIG được chọn sau khi xem lưới T6 trên cùng mẫu ngoài mẫu — khi viết bài cần nêu rõ, và trình bày cả lưới T6 để tránh nghi ngờ chọn tham số theo kết quả.
- **Hạn chế dữ liệu:** 19 mã hủy niêm yết thiếu giá; lợi suất sau ngày hủy niêm yết = 0; giá VCI làm tròn 100đ.
- **Tái lập:** `SEED = 0`; phiên bản thư viện ở `results/ENV_CONFIG.json`.